In [1]:
import os 

os.getcwd()

'/home/leostre/Рабочий стол/py-boost/experiments'

In [2]:
COMPUTATIONAL_METRICS = ['duration_seconds', 'inference_time', 'mean_leaves', 'mean_nodes', 'ntrees',
        'train_time', 'get_weights_avg_time', 'get_indexers_total_time', 'get_weights_calls', 'get_weights_total_time, get_indexers_avg_time', 'get_indexers_calls']
DETAILED_COMPUTATIONAL = ['get_weights_avg_time', 'get_indexers_total_time', 'get_weights_calls', 'get_weights_total_time, get_indexers_avg_time', 'get_indexers_calls']

In [3]:
def filter_df(df, rule):
    for k, v in rule.items():
        if k not in df.columns:
            continue
        df = df[df[k] == v]
    return df

In [29]:
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def get_all_runs_data(experiment_names=None, include_artifacts=False):
    """
    Extract all runs data from MLflow experiments into a comprehensive DataFrame
    
    Args:
        experiment_names: List of experiment names or None for all experiments
        include_artifacts: Whether to include artifact URIs
    
    Returns:
        DataFrame with all runs data
    """
    client = MlflowClient()
    
    # Get experiments
    if experiment_names is None:
        experiments = client.search_experiments()
    else:
        experiments = [client.get_experiment_by_name(name) for name in experiment_names]
        experiments = [exp for exp in experiments if exp is not None]
    
    all_runs_data = []
    
    for experiment in tqdm(experiments, desc="Processing experiments"):
        experiment_id = experiment.experiment_id
        experiment_name = experiment.name
        
        print(f"Processing experiment: {experiment_name}")
        
        # Get all runs for this experiment
        runs = client.search_runs(
            experiment_ids=[experiment_id],
            max_results=10000  # Adjust if you have more runs
        )
        
        for run in tqdm(runs, desc=f"Runs in {experiment_name}", leave=False):
            run_data = {
                'run_id': run.info.run_id,
                'experiment_id': experiment_id,
                'experiment_name': experiment_name,
                'run_name': run.data.tags.get('mlflow.runName', ''),
                'status': run.info.status,
                'start_time': pd.to_datetime(run.info.start_time, unit='ms'),
                'end_time': pd.to_datetime(run.info.end_time, unit='ms') if run.info.end_time else None,
                'duration_seconds': (run.info.end_time - run.info.start_time) / 1000.0 if run.info.end_time and run.info.start_time else None,
            }
            
            # Add parameters
            for key, value in run.data.params.items():
                run_data[f'param_{key}'] = value
            
            # Add metrics
            for key, value in run.data.metrics.items():
                run_data[f'metric_{key}'] = value
            
            # Add tags
            for key, value in run.data.tags.items():
                if key not in ['mlflow.runName', 'mlflow.user']:
                    run_data[f'tag_{key}'] = value
            
            # Add artifact location if requested
            if include_artifacts:
                run_data['artifact_uri'] = run.info.artifact_uri
            
            all_runs_data.append(run_data)
    
    return pd.DataFrame(all_runs_data)

def get_baselines(path, version):
    start_dir = os.getcwd()
    os.chdir(path)
    df = get_all_runs_data(['baselines_' + str(version)])
    os.chdir(start_dir)
    return df 


In [30]:
# bsln = get_baselines('/home/leostre/Рабочий стол/py-boost/5.2', '5.2')

In [31]:
EXCLUDE = {'experiment_id', 'experiment_name', 'status', 'start_time', 'end_time', 'run_name', 'tag_mlflow.source.name', 'tag_mlflow.source.git.commit',
       'tag_mlflow.source.type', 'param_error', 'tag_status',
       'param_total_runs', 'param_successful_runs', 'mean_f1', 'mean_accuracy', 'param_n_splits', 'param_n_successful_folds'}

def filter_data(data):
    after_exclusion_by_name = [
        col for col in data.columns if col not in EXCLUDE
    ]
    print(after_exclusion_by_name)
    statistics = ('mean', 'max', 'min', 'std', 'median')
    after_exclusion_agg = [
        col for col in after_exclusion_by_name if 'leaves' in col or 'nodes' in col or
        'tree' in col or
        not any(statistic in col for statistic in statistics) and not 'metric_fold' in col or col in ('param_stabilization_threshold', 'param_smoothing_alpha')
    ]
    filtered_data = data[after_exclusion_agg 
                        #  + ['metric_std_num_trees', 'metric_mean_num_trees',]
                         ]
    filtered_pivot = filtered_data.rename(columns={col: col.removeprefix('param_') for col in filtered_data.columns})
    return filtered_pivot

def melt_metrics(df):
    # Identify metric columns
    metric_cols = [col for col in df.columns if col.startswith('metric_')]
    print(metric_cols)
    
    # Identify ID columns (all non-metric columns)
    id_cols = [col for col in df.columns if not col.startswith('metric_')]
    print(id_cols)
    
    # Melt using pandas melt (more control)
    melted_df = df.melt(
        id_vars=id_cols,
        value_vars=metric_cols,
        var_name='metric_fold',
        value_name='value'
    )
    
    # Extract metric name and fold number using regex pattern
    pattern = r'metric_(.+?)_fold_(\d+)$'
    extracted = melted_df['metric_fold'].str.extract(pattern)
    
    # Create new columns
    melted_df['metric'] = extracted[0]
    melted_df['fold'] = extracted[1]
    
    # For metrics without fold numbers (like 'metric_total_training_time')
    # Fill NaN metric names with the original string without 'metric_' prefix
    mask = melted_df['metric'].isna()
    melted_df.loc[mask, 'metric'] = melted_df.loc[mask, 'metric_fold'].str.replace('metric_', '')
    
    # Drop the temporary column and clean up
    melted_df = melted_df.drop('metric_fold', axis=1)
    melted_df = melted_df.reset_index(drop=True)
    
    return melted_df


In [32]:
def flatten_index(df):
    columns = df.columns
    if isinstance(columns, pd.MultiIndex):
        columns = [c[0] if not c[1] else c[1] for c in columns] 
        columns = [c if c != 'mean' else 'value' for c in columns]
    df.columns = columns  

In [33]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
def comprehensive_statistical_test(df1: pd.DataFrame, df2: pd.DataFrame, value_cols=None):
    """
    Comprehensive statistical testing between two DataFrames
    
    Parameters:
    - value_cols: List of value columns to test (default: ['value', 'duration_seconds'])
    """
    if value_cols is None:
        value_cols = ['value', 'duration_seconds']
    
    group_cols = ['dataset', 'lr', 'subsample', 'sketch_method', 'sketch_outputs', 'metric']
    
    # Merge DataFrames
    merged = pd.merge(
        df1, 
        df2, 
        on=group_cols,
        suffixes=('_df1', '_df2'),
        how='inner'
    )
    
    all_results = {}
    
    for value_col in value_cols:
        if f"{value_col}_df1" not in merged.columns or f"{value_col}_df2" not in merged.columns:
            continue
            
        results = []
        
        for metric in merged['metric'].unique():
            metric_data = merged[merged['metric'] == metric]
            
            if len(metric_data) < 2:
                continue
                
            # Extract values
            values_df1 = metric_data[f"{value_col}_df1"].values
            values_df2 = metric_data[f"{value_col}_df2"].values
            
            # Paired t-test
            t_stat, p_value = stats.ttest_rel(values_df1, values_df2)
            
            # Calculate statistics
            differences = values_df1 - values_df2
            mean_diff = np.mean(differences)
            std_diff = np.std(differences, ddof=1)
            cohens_d = mean_diff / std_diff if std_diff > 0 else 0
            
            n = len(differences)
            se_diff = std_diff / np.sqrt(n)
            ci_low = mean_diff - 1.96 * se_diff
            ci_high = mean_diff + 1.96 * se_diff
            
            results.append({
                'metric': metric,
                'value_column': value_col,
                'n_pairs': n,
                'mean_df1': np.mean(values_df1),
                'mean_df2': np.mean(values_df2),
                'mean_difference': mean_diff,
                'std_difference': std_diff,
                't_statistic': t_stat,
                'p_value': p_value,
                'cohens_d': cohens_d,
                'ci_low': ci_low,
                'ci_high': ci_high,
                'significant_0.05': p_value < 0.05
            })
        
        results_df = pd.DataFrame(results)
        
        # Adjust p-values
        if not results_df.empty:
            rejected, pvals_corrected, _, _ = multipletests(
                results_df['p_value'].values, 
                alpha=0.05, 
                method='fdr_bh'
            )
            results_df['p_value_adj'] = pvals_corrected
            results_df['significant_adj'] = rejected
        
        all_results[value_col] = results_df
    
    return all_results, merged

In [34]:
# def statistical_test()

import scipy.stats as stats

test_fn = lambda x, y: stats.ttest_rel(x, y).pvalue

test_fn = lambda x, y: stats.ttest_ind_from_stats(
    np.mean(x), np.std(x), len(x), np.mean(y), np.std(y), len(y)
).pvalue

def ttest(df1, df2, p=0.05):
    gr_cols = ['dataset', 'lr', 'subsample',
       'sketch_method', 'sketch_outputs', 'metric',]
    to_cat = []
    for df in [df1, df2]:
        gr_df = df.groupby(gr_cols)['value'].aggregate(list)
        to_cat.append(gr_df.map(np.array))
    result = pd.Series([test_fn(x1, x2) for x1, x2 in zip(*to_cat)], index=gr_df.index) < p
    return result



## Compare experiments

In [35]:
# DATASET = [
#     # 'mediamill',
#     'mnist',
#     'cifar10',
#     # 'yeast', 
#         #    'age_prediction'
#         #    'birds', 
#         #    'genbase'
#     # 'mbd',
# ]


# dfs = [get_all_runs_data([exp]) for exp in EXPS]

# processed = {}
# for exp, df in zip(EXPS, dfs):
#     df = df[df.param_dataset.isin(DATASET)]
#     fp_df = filter_data(df)
#     mlt_df = melt_metrics(fp_df)
#     agg_mtrs = mlt_df.groupby(['dataset', 'sketch_method', 'sketch_outputs', 'subsample', 'lr',
#                                'stabilization_threshold',
#                                 'smoothing_alpha',
#                                  'metric']).agg({'value': 'mean'})#.reset_index()
#     processed[exp] = (agg_mtrs)




In [36]:
DATASET = [
    # 'mediamill',
    # 'mnist',
    # 'cifar10',
    # 'yeast', 
        #    'age_prediction'
        #    'birds', 
        #    'genbase'
    # 'mbd',
]

EXPS = [
    'sigmoid_5.2.1',
    'hyperbolic_5.2.1',
    'baselines_5.2.1',
    'lgbm_5.2.1',
    'xgboost_5.2.1',
    # 'sigmoid_5.2',
]
TO_INT = ['sketch_outputs']
TO_FLOAT = ['smoothing_alpha', 'stabilization_threshold', 'subsample', 'lr', ]

def process_mlruns(processed, EXPS, DATASET, path='.'):
    curdir = os.getcwd()
    os.chdir(path)
    dfs = [get_all_runs_data([exp]) for exp in EXPS] 

    version = None # '5.2'
    if version:
        dfs += [get_baselines(f'../{version}', version)]
        EXPS += [f'baselines_{version}']

    for exp, df in zip(EXPS, dfs):
        if DATASET:
            df = df[df.param_dataset.isin(DATASET)]
        df = df.rename(columns={'param_learning_rate': 'param_lr'})
        print(exp, df.shape)
        df = filter_data(df)
        mlt_df = melt_metrics(df)
        print(exp, mlt_df.shape)
        agg_mtrs = mlt_df.groupby(['dataset',
                                    *(['sketch_method', 'sketch_outputs'] if 'sketch_method' in mlt_df.columns else []), 
                                    *(['smoothing_alpha', 'stabilization_threshold'] if 'smoothing_alpha' in mlt_df.columns else []),
                                    'subsample', 'lr', 'metric', ]).agg({'value': ['mean', 'std']})#.reset_index()
        flatten_index(agg_mtrs)
        for c in TO_FLOAT:
            if not c in agg_mtrs:
                continue
            agg_mtrs[c] = agg_mtrs[c].astype(float)
        for c in TO_INT:
            if not c in agg_mtrs:
                continue
            agg_mtrs[c] = agg_mtrs[c].astype(int)
        agg_mtrs['experiment'] = exp
        processed[exp] = (agg_mtrs).reset_index()
    os.chdir(curdir)
    return processed

# dfs = [get_all_runs_data([exp]) for exp in EXPS] 

# version = None # '5.2'
# if version:
#     dfs += [get_baselines(f'../{version}', version)]
#     EXPS += [f'baselines_{version}']

# processed = {}
# for exp, df in zip(EXPS, dfs):
#     if DATASET:
#         df = df[df.param_dataset.isin(DATASET)]
#     df = df.rename(columns={'param_learning_rate': 'param_lr'})
#     df = filter_data(df)
#     mlt_df = melt_metrics(df)
#     agg_mtrs = mlt_df.groupby(['dataset',
#                                 *(['sketch_method', 'sketch_outputs'] if 'sketch_method' in mlt_df.columns else []), 
#                                 *(['smoothing_alpha', 'stabilization_threshold'] if 'smoothing_alpha' in mlt_df.columns else []),
#                                 'subsample', 'lr', 'metric', ]).agg({'value': ['mean', 'std']})#.reset_index()
#     flatten_index(agg_mtrs)
#     agg_mtrs['experiment'] = exp
#     processed[exp] = (agg_mtrs)

In [12]:
os.getcwd()

'/home/leostre/Рабочий стол/py-boost/experiments'

In [14]:
processed = process_mlruns({}, [
    'sigmoid_5.2.1',
    'hyperbolic_5.2.1',
    'baselines_5.2.1',
    'lgbm_5.2.1',
    'xgboost_5.2.1',
    # 'sigmoid_5.2',
], DATASET, '/home/leostre/Рабочий стол/py-boost/analysis')

Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: sigmoid_5.2.1


Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: hyperbolic_5.2.1


Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing experiment: lgbm_5.2.1


Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: xgboost_5.2.1


Processing experiments: 100%|██████████| 1/1 [00:00<00:00, 14.49it/s]


sigmoid_5.2.1 (847, 176)
['run_id', 'duration_seconds', 'param_es', 'param_max_bin', 'param_dataset', 'param_sketch_params', 'param_ntrees', 'param_lr', 'param_colsample', 'param_min_data_in_bin', 'param_subsample', 'param_min_gain_to_split', 'param_stabilization_threshold', 'param_sketch_method', 'param_callbacks', 'param_gd_steps', 'param_lambda_l2', 'param_quantization', 'param_use_hess', 'param_verbose', 'param_sketch_outputs', 'param_loss', 'param_smoothing_alpha', 'param_min_data_in_leaf', 'param_quant_sample', 'param_max_depth', 'param_metric', 'param_seed', 'metric_train_time_fold_3', 'metric_f1_fold_2', 'metric_train_time_fold_2', 'metric_recall_fold_0', 'metric_inference_time_fold_2', 'metric_exact_match_fold_0', 'metric_multiclass_logloss_fold_2', 'metric_f1_macro_fold_3', 'metric_ntrees_fold_2', 'metric_ntrees_fold_0', 'metric_f1_micro_fold_1', 'metric_f1_micro_fold_2', 'metric_f1_micro_fold_3', 'metric_precision_fold_0', 'metric_mean_nodes_fold_3', 'metric_f1_micro_fold_0'

In [37]:
processed = process_mlruns(processed,  [
    'hyperbolic_5.2',
    'baselines_5.2',
    'sigmoid_5.2',
], DATASET, '/home/leostre/Рабочий стол/py-boost/5.2')

Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: hyperbolic_5.2


Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: baselines_5.2


Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: sigmoid_5.2


Processing experiments: 100%|██████████| 1/1 [00:00<00:00,  1.79it/s]

hyperbolic_5.2 (300, 122)
['run_id', 'duration_seconds', 'param_es', 'param_max_bin', 'param_dataset', 'param_sketch_params', 'param_ntrees', 'param_lr', 'param_colsample', 'param_min_data_in_bin', 'param_subsample', 'param_min_gain_to_split', 'param_stabilization_threshold', 'param_sketch_method', 'param_callbacks', 'param_gd_steps', 'param_lambda_l2', 'param_quantization', 'param_use_hess', 'param_verbose', 'param_sketch_outputs', 'param_loss', 'param_smoothing_alpha', 'param_min_data_in_leaf', 'param_quant_sample', 'param_max_depth', 'param_metric', 'param_seed', 'metric_get_indexers_calls_fold_2', 'metric_train_time_fold_3', 'metric_f1_fold_2', 'metric_train_time_fold_2', 'metric_get_weights_total_time_fold_1', 'metric_recall_fold_0', 'metric_get_weights_calls_fold_3', 'metric_inference_time_fold_2', 'metric_get_indexers_total_time_fold_3', 'metric_ntrees_fold_2', 'metric_get_indexers_avg_time_fold_1', 'metric_ntrees_fold_0', 'metric_get_weights_avg_time_fold_0', 'metric_precision_

In [15]:
import matplotlib.pyplot as plt 
import seaborn as sns 


def metric_heatmaps(df, x, y):
    # Pivot the data to create a matrix for each metric
    pivot_dfs = {}
    metrics = df['metric'].unique()

    for metric in metrics:
        # Filter for the specific metric
        metric_df = df[df['metric'] == metric].copy()
        
        # Pivot to create matrix
        pivot = metric_df.pivot_table(
            index=x, 
            columns=y, 
            values='value',
            aggfunc='mean'
        )
        pivot_dfs[metric] = pivot

    # Create heatmaps for each metric
    fig, axes = plt.subplots((len(metrics) + 3) // 4, 4, figsize=(20, 10))
    axes = axes.flatten()

    for i, (metric, pivot_df) in enumerate(pivot_dfs.items()):
        if i < len(axes):
            sns.heatmap(pivot_df, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[i],
                    cbar_kws={'label': metric})
            axes[i].set_title(f'{metric}')
            axes[i].set_xlabel(y)
            axes[i].set_ylabel(x)

    # Hide any unused subplots
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()




In [16]:
metric_heatmaps(
    processed['sigmoid_5.2'].reset_index().groupby(['smoothing_alpha', 'stabilization_threshold', 'metric'])['value'].aggregate('mean').reset_index(), 'smoothing_alpha', 'stabilization_threshold'
)

KeyError: 'sigmoid_5.2'

## setting up the other HPs

sthreshold for correct comparison

to create baselines for corruption experiments

In [17]:
import numpy as np 


def compare_prepare(modified, base, filter, method='standard', include_detailed=False):
    modified_data = processed[modified].reset_index()
    if base:
        basic = processed[base].reset_index()
    else:
        basic = modified_data.copy()
        basic.loc[:, :] = np.zeros(modified_data.shape)

    hps = set(modified_data.columns) & set(basic.columns)
    for c in ['std', 'index', 'experiment']: 
        hps.discard(c)
        if c in modified_data.columns:
            modified_data = modified_data.drop(c, axis=1)
            basic = basic.drop(c, axis=1)

    hps.discard('value'); 
    hps = list(hps)
    difference = pd.merge(modified_data, basic, on=hps, suffixes=['_m', '_b'])
    assert difference.shape[0] > 0
    metrics = difference['metric']
    difference['value'] = (difference['value_m'] - difference['value_b']) * 100
    
    if include_detailed:
        is_computational = metrics.isin(COMPUTATIONAL_METRICS).values
    else:
        is_computational = (metrics.isin(COMPUTATIONAL_METRICS) & ~metrics.isin(DETAILED_COMPUTATIONAL)).values
    difference.loc[is_computational, 'value'] = difference.loc[is_computational, 'value'] / difference.loc[is_computational, 'value_b']
    difference.drop(['value_b', 'value_m'], axis=1, inplace=True)
    for c in TO_FLOAT:
        if not c in difference:
            continue
        difference[c] = difference[c].astype(float)
    for c in TO_INT:
        if not c in difference:
            continue
        difference[c] = difference[c].astype(int)
    difference = filter_df(difference, filter)
    difference = difference.set_index(hps).sort_index().reset_index()
    # difference.sort_values(hps, inplace=True)
    return difference


Это хорошая визуализация отдельно по метрикам, но снизу есть версия плотли

to create archives with graphics

In [18]:
# import shutil
# import os

# # Basic usage
# shutil.make_archive('graphs0.05subsample', 'zip', 'GRAPHS')

# # With full path
# # shutil.make_archive('/path/to/archive_name', 'zip', '/path/to/source_folder')

## Main comparison

In [19]:

import plotly.express as px

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# Assuming your dataframe is called `df`
# Required columns: 'sketch_method', 'subsample', 'sketch_outputs', and some value column
def create_plot_for_metric(df):
    # Create subplots - one column, multiple rows
    sketch_methods = df['sketch_method'].unique()
    fig = make_subplots(
        rows=len(sketch_methods), 
        cols=1,
        subplot_titles=sketch_methods,
        vertical_spacing=0.08
    )

    # Add a bar plot for each sketch_method
    for i, method in enumerate(sketch_methods, 1):
        method_data = df[df['sketch_method'] == method]
        
        # Get unique combinations of subsample and sketch_outputs
        subsample_levels = method_data['subsample'].unique()
        sketch_outputs = method_data['sketch_outputs'].unique()
        
        # Create bars for each sketch_outputs within each subsample
        for j, output in enumerate(sketch_outputs):
            output_data = method_data[method_data['sketch_outputs'] == output]
            
            # You'll need to aggregate your data appropriately
            # This assumes you have a 'value' column to plot
            fig.add_trace(
                go.Bar(
                    x=output_data['subsample'],
                    y=output_data['value'],  # Replace with your actual value column
                    name=str(output),
                    legendgroup=str(output),
                    showlegend=(i == 1),  # Only show legend for first subplot
                    marker_color=px.colors.qualitative.Set1[j % len(px.colors.qualitative.Set1)]
                ),
                row=i, col=1
            )

    # Update layout
    fig.update_layout(
        height=300 * len(sketch_methods),
        title_text=f"Sketch Methods Analysis | Metric: {df.metric.iloc[0]}",
        barmode='group',  # This groups bars by subsample with different colors for sketch_outputs
        showlegend=True
    )
    return fig

    

def samplings(df, name=None):
    metrics = df.metric.unique()
    for metric in metrics:
        subset = df[df.metric == metric]

        fig = create_plot_for_metric(subset)
        from pathlib import Path
        if name:
            fig.write_image(Path('/home/leostre/Рабочий стол/py-boost/experiments/cifar10')/ f'{metric}_{name}.jpg')
        fig.show()


# samplings(difference)
        

In [20]:
d = compare_prepare('hyperbolic_5.2', 'baselines_5.2', {'dataset': 'cifar10', 'sketch_method': 'topk', 'lr': 0.1, 
                                                     'smoothing_alpha': 0.9, 'stabilization_threshold': 1.0
                                                     })

samplings(d, 'hyperbolic_over_baseline')


KeyError: 'hyperbolic_5.2'

In [21]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def plot_metric_heatmaps(df, dataset=None, sketch_method=None, lr=None, figsize=(15, 10)):
    """
    Plot separate heatmaps for each metric
    Grid: subsample (rows) x sketch_outputs (columns)
    """
    # Filter data if specific parameters provided
    filtered_df = df.copy()
    if dataset:
        filtered_df = filtered_df[filtered_df.dataset == dataset]
    if sketch_method:
        filtered_df = filtered_df[filtered_df.sketch_method == sketch_method]
    if lr:
        filtered_df = filtered_df[filtered_df.lr == lr]
    
    metrics = filtered_df['metric'].unique()
    metrics = [mtr for mtr in metrics if not mtr in ('mean_nodes',)]
    metric =  [mtr for mtr in metrics if mtr not in COMPUTATIONAL_METRICS] + [mtr for mtr in metrics if mtr in COMPUTATIONAL_METRICS]
    n_metrics = len(metrics)
    
    # Calculate grid layout
    n_cols = min(4, n_metrics)
    n_rows = (n_metrics + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(figsize[0], figsize[1] * n_rows / 2))
    axes = axes.flatten() if n_metrics > 1 else [axes]
    
    for i, metric in enumerate(metrics):
        
        if i >= len(axes):
            break
            
        ax = axes[i]
        metric_data = filtered_df[filtered_df['metric'] == metric]
        
        # Create pivot table: subsample vs sketch_outputs
        pivot_table = metric_data.pivot_table(
            values='value',
            index='subsample',
            columns='sketch_outputs',
            aggfunc='mean'  # Use mean if multiple values exist
        )
        
        # Sort for better visualization
        pivot_table = pivot_table.sort_index(ascending=False)  # Reverse subsample order
        if metric in COMPUTATIONAL_METRICS:
            pivot_table = pivot_table.round(1)
        
        # Create heatmap
        sns.heatmap(
            pivot_table,
            ax=ax,
            cmap='viridis',
            annot=True,
            fmt='.3f' if metric not in COMPUTATIONAL_METRICS else '.1f',
            cbar_kws={'label': 'Value'},
            linewidths=0.5,
            linecolor='gray'
        )
        
        ax.set_title(f'Metric: {metric}', fontsize=14, fontweight='bold', pad=20)
        ax.set_xlabel('Sketch Outputs', fontsize=12)
        ax.set_ylabel('Subsample', fontsize=12)
        
        # Rotate x labels for better readability
        ax.tick_params(axis='x', rotation=45)
        ax.tick_params(axis='y', rotation=0)
    
    # Hide empty subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)
    
    # Create super title
    title_parts = []
    if dataset: title_parts.append(f"Dataset: {dataset}")
    if sketch_method: title_parts.append(f"Method: {sketch_method}")
    if lr: title_parts.append(f"LR: {lr}")
    super_title = " | ".join(title_parts) if title_parts else "All Configurations"
    
    plt.suptitle(f'Performance Heatmaps\n{super_title}', fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()


In [22]:
# d = compare_prepare('baselines_3.1', None, {"lr": '0.005', 'dataset': 'age_prediction', 'sketch_method': 'topk'})
# Usage
plot_metric_heatmaps(processed['baselines_5.2'].reset_index(), dataset='mnist', lr='0.005', sketch_method='topk' )

KeyError: 'baselines_5.2'

In [23]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np

def create_metric_scatter_long(df, x_metric, y_metric, signature_flag=False):
    """
    Create a scatter plot from long-format data with two specified metrics.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe in long format with 'metric' and 'value' columns
    x_metric : str
        Name of the metric to plot on X-axis
    y_metric : str
        Name of the metric to plot on Y-axis
    signature_flag : bool
        If True, adds straight lines at zero positions on both axes
    
    Returns:
    --------
    plotly.graph_objects.Figure
    """
    
    # Pivot the data to get metrics as columns
    # Identify all columns that are not 'metric' or 'value'
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    
    # Pivot the dataframe
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'  # Assuming one value per combination
    ).reset_index()
    
    # Filter to keep only rows where both metrics exist
    df_plot = df_pivoted.dropna(subset=[x_metric, y_metric])
    
    # Create a unique combination identifier for coloring
    # List all categorical columns (excluding the metrics)
    categorical_cols = id_vars
    
    # Create a combined category for coloring
    df_plot['color_category'] = df_plot[categorical_cols].astype(str).agg(' | '.join, axis=1)
    
    # Create the scatter plot
    fig = px.scatter(
        df_plot,
        x=x_metric,
        y=y_metric,
        color='color_category',
        hover_data=categorical_cols,
        title=f'Scatter Plot: {x_metric} vs {y_metric}',
        labels={x_metric: x_metric, y_metric: y_metric, 'color_category': 'Configuration'}
    )
    
    # Add zero lines if signature_flag is True
    if signature_flag:
        fig.add_vline(x=0, line_width=1.5, line_dash="dash", 
                      line_color="gray", opacity=0.7)
        fig.add_hline(y=0, line_width=1.5, line_dash="dash", 
                      line_color="gray", opacity=0.7)
    
    # Improve layout
    fig.update_layout(
        plot_bgcolor='white',
        xaxis=dict(
            title=x_metric,
            gridcolor='lightgray',
            showgrid=True,
            zeroline=False
        ),
        yaxis=dict(
            title=y_metric,
            gridcolor='lightgray',
            showgrid=True,
            zeroline=False
        ),
        legend=dict(
            title="Configuration",
            itemsizing='constant',
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.02
        ),
        hovermode='closest'
    )
    
    return fig


def create_metric_scatter_long_advanced(df, x_metric, y_metric, signature_flag=False):
    """
    Advanced version for long-format data with more customization.
    """
    # Pivot the data
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Filter to keep only rows with both metrics
    df_plot = df_pivoted.dropna(subset=[x_metric, y_metric])
    
    # Create unique color categories
    categorical_cols = id_vars
    df_plot['color_id'] = df_plot[categorical_cols].astype(str).agg(' | '.join, axis=1)
    unique_configs = df_plot['color_id'].unique()
    
    # Create figure
    fig = go.Figure()
    
    # Add traces for each unique configuration
    for config in unique_configs:
        config_data = df_plot[df_plot['color_id'] == config]
        
        # Create hover text with all configuration details
        hover_text = []
        for _, row in config_data.iterrows():
            hover_info_lines = ["<b>Configuration:</b>"]
            for col in categorical_cols:
                hover_info_lines.append(f"{col}: {row[col]}")
            hover_info_lines.extend([
                "",
                f"<b>{x_metric}: {row[x_metric]:.4f}</b>",
                f"<b>{y_metric}: {row[y_metric]:.4f}</b>"
            ])
            hover_text.append('<br>'.join(hover_info_lines))
        
        fig.add_trace(go.Scatter(
            x=config_data[x_metric],
            y=config_data[y_metric],
            mode='markers',
            name=config[:40] + '...' if len(config) > 40 else config,
            text=hover_text,
            hoverinfo='text',
            marker=dict(
                size=10,
                opacity=0.7,
                line=dict(width=1, color='DarkSlateGrey')
            ),
            showlegend=True
        ))
    
    # Add zero lines if requested
    if signature_flag:
        fig.add_shape(
            type="line", x0=0, x1=0, 
            y0=df_plot[y_metric].min(), y1=df_plot[y_metric].max(),
            line=dict(color="gray", width=1.5, dash="dash")
        )
        fig.add_shape(
            type="line", y0=0, y1=0,
            x0=df_plot[x_metric].min(), x1=df_plot[x_metric].max(),
            line=dict(color="gray", width=1.5, dash="dash")
        )
    
    # Update layout
    fig.update_layout(
        title=f'{x_metric} vs {y_metric}',
        xaxis_title=x_metric,
        yaxis_title=y_metric,
        plot_bgcolor='white',
        xaxis=dict(
            gridcolor='lightgray',
            showgrid=True,
            zeroline=False,
            tickformat='.3f'
        ),
        yaxis=dict(
            gridcolor='lightgray',
            showgrid=True,
            zeroline=False,
            tickformat='.3f'
        ),
        legend=dict(
            title="Configuration",
            itemsizing='constant',
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.02,
            font=dict(size=9)
        ),
        hoverlabel=dict(
            bgcolor="white",
            font_size=11,
            font_family="monospace"
        ),
        margin=dict(r=200)  # Extra margin for legend
    )
    
    return fig


def list_available_metrics(df):
    """
    Helper function to list all available metrics in the long-format dataframe.
    """
    if 'metric' in df.columns:
        metrics = df['metric'].unique()
        print("Available metrics:")
        for metric in metrics:
            print(f"  - {metric}")
        return metrics
    else:
        print("No 'metric' column found in dataframe")
        return []


# Example usage:
# Assuming your dataframe 'df' has columns: dataset, sketch_method, sketch_outputs, 
# smoothing_alpha, stabilization_threshold, subsample, lr, metric, value

# List available metrics
# available_metrics = list_available_metrics(df)

# Create scatter plot comparing two metrics
# fig = create_metric_scatter_long(df, x_metric='accuracy', y_metric='loss', signature_flag=True)
# fig.show()

# Or with advanced version
# fig_advanced = create_metric_scatter_long_advanced(df, x_metric='accuracy', y_metric='loss', signature_flag=True)
# fig_advanced.show()

# If you want to filter specific configurations before plotting:
def filter_and_plot(df, x_metric, y_metric, filters=None, signature_flag=False):
    """
    Apply filters to the dataframe before creating the scatter plot.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe in long format
    x_metric, y_metric : str
        Metrics to plot
    filters : dict
        Dictionary of column: value filters to apply
    signature_flag : bool
        Add zero lines or not
    
    Returns:
    --------
    plotly.graph_objects.Figure
    """
    df_filtered = df.copy()
    
    if filters:
        for col, val in filters.items():
            if col in df_filtered.columns:
                df_filtered = df_filtered[df_filtered[col] == val]
    
    return create_metric_scatter_long(df_filtered, x_metric, y_metric, signature_flag)


# Example with filtering:
# fig_filtered = filter_and_plot(
#     df, 
#     x_metric='accuracy', 
#     y_metric='loss',
#     filters={'dataset': 'cifar10', 'lr': '0.001'},
#     signature_flag=True
# )
# fig_filtered.show()

In [26]:
d = compare_prepare('hyperbolic_5.2.1', 'baselines_5.2.1', {})

In [24]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from ipywidgets import interact, widgets, VBox, HBox, Output
from IPython.display import display, clear_output

def create_interactive_metric_scatter(df, x_metric, y_metric, signature_flag=False):
    """
    Create an interactive scatter plot with filtering widgets and flexible coloration.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe in long format
    x_metric, y_metric : str
        Metrics to plot
    signature_flag : bool
        Add zero lines or not
    """
    
    # Pivot the data
    df = df.copy()
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Identify categorical columns for filtering
    categorical_cols = [col for col in id_vars if df[col].dtype == 'object' or df[col].nunique() < 20]
    
    # Create filters dictionary
    filters = {}
    
    def create_filter_widget(col):
        """Create a filter widget for a column"""
        unique_vals = sorted(df_pivoted[col].dropna().unique())
        return widgets.SelectMultiple(
            options=unique_vals,
            description=col,
            layout=widgets.Layout(width='300px'),
            style={'description_width': 'initial'}
        )
    
    # Create filter widgets
    filter_widgets = {col: create_filter_widget(col) for col in categorical_cols}
    
    # Create color by selector
    color_by_widget = widgets.Dropdown(
        options=['None'] + categorical_cols + ['color_category_auto'],
        value='color_category_auto',
        description='Color by:',
        layout=widgets.Layout(width='300px'),
        style={'description_width': 'initial'}
    )
    
    # Create output widget for the plot
    output_widget = Output()
    
    def update_plot(change=None):
        """Update function called when filters or color selection changes"""
        with output_widget:
            clear_output(wait=True)
            
            # Apply filters
            filtered_df = df_pivoted.copy()
            for col, widget in filter_widgets.items():
                if widget.value:
                    filtered_df = filtered_df[filtered_df[col].isin(widget.value)]
            
            if len(filtered_df) == 0:
                print("No data matches the selected filters")
                return
            
            # Determine coloring
            color_by = color_by_widget.value
            
            if color_by == 'None':
                # All points same color
                fig = px.scatter(
                    filtered_df,
                    x=x_metric,
                    y=y_metric,
                    title=f'{x_metric} vs {y_metric}',
                    labels={x_metric: x_metric, y_metric: y_metric}
                )
                fig.update_traces(marker=dict(color='blue', opacity=0.7))
            elif color_by == 'color_category_auto':
                # Color by unique combination of all categorical columns
                filtered_df['color_category'] = filtered_df[categorical_cols].astype(str).agg(' | '.join, axis=1)
                fig = px.scatter(
                    filtered_df,
                    x=x_metric,
                    y=y_metric,
                    color='color_category',
                    title=f'{x_metric} vs {y_metric}',
                    labels={x_metric: x_metric, y_metric: y_metric, 'color_category': 'Configuration'}
                )
            else:
                # Color by specific column
                fig = px.scatter(
                    filtered_df,
                    x=x_metric,
                    y=y_metric,
                    color=color_by,
                    title=f'{x_metric} vs {y_metric}',
                    labels={x_metric: x_metric, y_metric: y_metric, color_by: color_by},
                    color_continuous_scale='Viridis' if filtered_df[color_by].dtype in ['float64', 'int64'] else 'Set1'
                )
            
            # Add zero lines if requested
            if signature_flag:
                fig.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
                fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
            
            # Update layout
            fig.update_layout(
                plot_bgcolor='white',
                xaxis=dict(
                    title=x_metric,
                    gridcolor='lightgray',
                    showgrid=True,
                    zeroline=False
                ),
                yaxis=dict(
                    title=y_metric,
                    gridcolor='lightgray',
                    showgrid=True,
                    zeroline=False
                ),
                hovermode='closest',
                height=600
            )
            
            fig.show()
    
    # Create filter UI
    filter_controls = []
    
    # Group filters in rows of 3
    filter_items = list(filter_widgets.items())
    for i in range(0, len(filter_items), 3):
        row = HBox([widget for _, widget in filter_items[i:i+3]])
        filter_controls.append(row)
    
    # Create control panel
    control_panel = VBox([
        widgets.HTML("<b>Filters:</b>"),
        *filter_controls,
        widgets.HTML("<br><b>Styling Options:</b>"),
        HBox([color_by_widget]),
        widgets.HTML("<br><b>Plot Controls:</b>"),
        widgets.Button(description='Reset All Filters', button_style='warning')
    ])
    
    # Reset button functionality
    reset_button = control_panel.children[-1]
    def reset_filters(b):
        for widget in filter_widgets.values():
            widget.value = []
        color_by_widget.value = 'color_category_auto'
    reset_button.on_click(reset_filters)
    
    # Attach observers
    for widget in filter_widgets.values():
        widget.observe(update_plot, names='value')
    color_by_widget.observe(update_plot, names='value')
    
    # Initial plot
    update_plot()
    
    # Display the complete interface
    display(VBox([control_panel, output_widget]))
    
    return filter_widgets, color_by_widget


def create_advanced_interactive_scatter(df, x_metric, y_metric, signature_flag=False):
    """
    More advanced version with additional features like:
    - Multiple color modes (categorical, numerical, custom)
    - Size encoding
    - Opacity control
    - Show/hide legend
    - Export data button
    """
    
    # Pivot the data
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Get all columns for selection
    categorical_cols = [col for col in id_vars if df[col].dtype == 'object' or df[col].nunique() < 20]
    numerical_cols = [col for col in id_vars if df[col].dtype in ['float64', 'int64'] and col not in categorical_cols]
    all_metrics = [col for col in df_pivoted.columns if col not in id_vars]
    
    # Create widgets
    filters = {}
    
    # Create filter widgets with multi-select
    filter_widgets = {}
    for col in categorical_cols:
        unique_vals = sorted(df_pivoted[col].dropna().unique())
        filter_widgets[col] = widgets.SelectMultiple(
            options=unique_vals,
            description=col[:15],
            layout=widgets.Layout(width='250px'),
            style={'description_width': 'initial'}
        )
    
    # Color by widget
    color_options = ['None', 'Auto (All columns)'] + categorical_cols + numerical_cols + all_metrics
    color_by_widget = widgets.Dropdown(
        options=color_options,
        value='Auto (All columns)',
        description='Color by:',
        layout=widgets.Layout(width='250px')
    )
    
    # Size by widget
    size_by_widget = widgets.Dropdown(
        options=['None'] + numerical_cols + all_metrics,
        value='None',
        description='Size by:',
        layout=widgets.Layout(width='250px')
    )
    
    # Opacity slider
    opacity_slider = widgets.FloatSlider(
        value=0.7,
        min=0.1,
        max=1.0,
        step=0.05,
        description='Opacity:',
        layout=widgets.Layout(width='250px')
    )
    
    # Point size slider
    point_size_slider = widgets.IntSlider(
        value=8,
        min=2,
        max=20,
        description='Point size:',
        layout=widgets.Layout(width='250px')
    )
    
    # Legend toggle
    legend_toggle = widgets.Checkbox(
        value=True,
        description='Show legend',
        layout=widgets.Layout(width='150px')
    )
    
    # Grid toggle
    grid_toggle = widgets.Checkbox(
        value=True,
        description='Show grid',
        layout=widgets.Layout(width='150px')
    )
    
    # Color scale for numerical data
    color_scale_widget = widgets.Dropdown(
        options=['Viridis', 'Plasma', 'Inferno', 'Magma', 'Cividis', 'Blues', 'Reds', 'Greens'],
        value='Viridis',
        description='Color scale:',
        layout=widgets.Layout(width='250px')
    )
    
    output_widget = Output()
    
    def update_plot(change=None):
        with output_widget:
            clear_output(wait=True)
            
            # Apply filters
            filtered_df = df_pivoted.copy()
            for col, widget in filter_widgets.items():
                if widget.value:
                    filtered_df = filtered_df[filtered_df[col].isin(widget.value)]
            
            if len(filtered_df) == 0:
                print("No data matches the selected filters")
                return
            
            # Determine coloring
            color_by = color_by_widget.value
            
            if color_by == 'None':
                fig = px.scatter(
                    filtered_df,
                    x=x_metric,
                    y=y_metric,
                    title=f'{x_metric} vs {y_metric}'
                )
                fig.update_traces(
                    marker=dict(
                        color='blue',
                        opacity=opacity_slider.value,
                        size=point_size_slider.value
                    )
                )
            elif color_by == 'Auto (All columns)':
                filtered_df['color_category'] = filtered_df[categorical_cols].astype(str).agg(' | '.join, axis=1)
                fig = px.scatter(
                    filtered_df,
                    x=x_metric,
                    y=y_metric,
                    color='color_category',
                    title=f'{x_metric} vs {y_metric}',
                    labels={'color_category': 'Configuration'}
                )
                fig.update_traces(
                    marker=dict(
                        opacity=opacity_slider.value,
                        size=point_size_slider.value
                    )
                )
            else:
                # Check if the color column is numerical or categorical
                is_numerical = filtered_df[color_by].dtype in ['float64', 'int64']
                
                if is_numerical and color_by not in categorical_cols:
                    # Numerical color
                    fig = px.scatter(
                        filtered_df,
                        x=x_metric,
                        y=y_metric,
                        color=color_by,
                        title=f'{x_metric} vs {y_metric}',
                        color_continuous_scale=color_scale_widget.value,
                        labels={color_by: color_by}
                    )
                    fig.update_traces(
                        marker=dict(
                            opacity=opacity_slider.value,
                            size=point_size_slider.value
                        )
                    )
                else:
                    # Categorical color
                    fig = px.scatter(
                        filtered_df,
                        x=x_metric,
                        y=y_metric,
                        color=color_by,
                        title=f'{x_metric} vs {y_metric}',
                        color_discrete_sequence=px.colors.qualitative.Set1,
                        labels={color_by: color_by}
                    )
                    fig.update_traces(
                        marker=dict(
                            opacity=opacity_slider.value,
                            size=point_size_slider.value
                        )
                    )
            
            # Apply size encoding if selected
            if size_by_widget.value != 'None':
                size_col = size_by_widget.value
                # Normalize sizes to range [5, 20]
                if size_col in filtered_df.columns:
                    size_vals = filtered_df[size_col].fillna(filtered_df[size_col].median())
                    min_size, max_size = size_vals.min(), size_vals.max()
                    if min_size != max_size:
                        normalized_sizes = 5 + (size_vals - min_size) / (max_size - min_size) * 15
                    else:
                        normalized_sizes = [10] * len(filtered_df)
                    
                    fig.update_traces(
                        marker=dict(
                            size=normalized_sizes,
                            sizemode='area',
                            sizeref=2.*max(normalized_sizes)/(40**2)
                        )
                    )
            
            # Add zero lines if requested
            if signature_flag:
                fig.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
                fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
            
            # Update layout based on toggles
            fig.update_layout(
                plot_bgcolor='white',
                xaxis=dict(
                    title=x_metric,
                    gridcolor='lightgray',
                    showgrid=grid_toggle.value,
                    zeroline=False
                ),
                yaxis=dict(
                    title=y_metric,
                    gridcolor='lightgray',
                    showgrid=grid_toggle.value,
                    zeroline=False
                ),
                hovermode='closest',
                height=600,
                showlegend=legend_toggle.value
            )
            
            # Add hover information
            hover_template = "<b>Configuration</b><br>"
            for col in categorical_cols:
                hover_template += f"{col}: %{{customdata[{categorical_cols.index(col)}]}}<br>"
            hover_template += f"<br><b>{x_metric}</b>: %{{x:.4f}}<br>"
            hover_template += f"<b>{y_metric}</b>: %{{y:.4f}}<br>"
            hover_template += "<extra></extra>"
            
            # Add customdata for hover
            fig.update_traces(
                customdata=filtered_df[categorical_cols],
                hovertemplate=hover_template
            )
            
            fig.show()
    
    # Create UI layout
    # Left panel with filters
    filter_panel = VBox([
        widgets.HTML("<b>Filters:</b>"),
        *[HBox([widget]) for widget in filter_widgets.values()]
    ])
    
    # Right panel with styling options
    style_panel = VBox([
        widgets.HTML("<b>Styling:</b>"),
        color_by_widget,
        size_by_widget,
        color_scale_widget,
        opacity_slider,
        point_size_slider,
        legend_toggle,
        grid_toggle
    ])
    
    # Main control panel
    control_panel = HBox([filter_panel, style_panel])
    
    # Add reset button
    reset_button = widgets.Button(description='Reset All', button_style='warning')
    
    def reset_all(b):
        for widget in filter_widgets.values():
            widget.value = []
        color_by_widget.value = 'Auto (All columns)'
        size_by_widget.value = 'None'
        opacity_slider.value = 0.7
        point_size_slider.value = 8
        legend_toggle.value = True
        grid_toggle.value = True
        color_scale_widget.value = 'Viridis'
    
    reset_button.on_click(reset_all)
    
    # Attach observers
    for widget in filter_widgets.values():
        widget.observe(update_plot, names='value')
    color_by_widget.observe(update_plot, names='value')
    size_by_widget.observe(update_plot, names='value')
    opacity_slider.observe(update_plot, names='value')
    point_size_slider.observe(update_plot, names='value')
    legend_toggle.observe(update_plot, names='value')
    grid_toggle.observe(update_plot, names='value')
    color_scale_widget.observe(update_plot, names='value')
    
    # Initial plot
    update_plot()
    
    # Display complete interface
    display(VBox([control_panel, reset_button, output_widget]))
    
    return filter_widgets, color_by_widget, size_by_widget


# Helper function to create a static version with dropdown filters using Plotly's built-in features
def create_static_with_filters(df, x_metric, y_metric, signature_flag=False):
    """
    Create a static Plotly figure with dropdown filters (no ipywidgets required)
    """
    # Pivot the data
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Create color category
    df_pivoted['color_category'] = df_pivoted[id_vars].astype(str).agg(' | '.join, axis=1)
    
    # Create figure
    fig = px.scatter(
        df_pivoted,
        x=x_metric,
        y=y_metric,
        color='color_category',
        title=f'{x_metric} vs {y_metric}',
        labels={'color_category': 'Configuration'}
    )
    
    # Add filters as dropdown menus
    categorical_cols = [col for col in id_vars if df[col].dtype == 'object' or df[col].nunique() < 20]
    
    buttons = []
    for col in categorical_cols:
        unique_vals = ['All'] + sorted(df_pivoted[col].dropna().unique())
        for val in unique_vals:
            buttons.append(
                dict(
                    method='restyle',
                    label=f'{col}: {val}',
                    args=[{'visible': [val == 'All' or df_pivoted[col].iloc[i] == val for i in range(len(df_pivoted))]}]
                )
            )
    
    fig.update_layout(
        updatemenus=[
            dict(
                buttons=buttons[:10],  # Limit to first 10 for performance
                direction="down",
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.1,
                xanchor="left",
                y=1.1,
                yanchor="top"
            ),
        ]
    )
    
    # Add zero lines if requested
    if signature_flag:
        fig.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
        fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
    
    fig.update_layout(
        plot_bgcolor='white',
        xaxis=dict(gridcolor='lightgray', showgrid=True, zeroline=False),
        yaxis=dict(gridcolor='lightgray', showgrid=True, zeroline=False),
        height=600
    )
    
    return fig


# Example usage:
# For interactive version (requires ipywidgets):
# create_interactive_metric_scatter(df, 'accuracy', 'loss', signature_flag=True)

# For advanced version with more controls:
# create_advanced_interactive_scatter(df, 'accuracy', 'loss', signature_flag=True)

# For static version with dropdowns (no ipywidgets needed):
# fig = create_static_with_filters(df, 'accuracy', 'loss', signature_flag=True)
# fig.show()

In [66]:
# def create_advanced_interactive_scatter_with_baseline(df, x_metric, y_metric, signature_flag=False):
#     """
#     Enhanced version with baseline comparison - with flexible hyperparameter matching.
#     Can automatically handle NaN values or manually select matching columns.
#     """
    
#     # Pivot the data
#     id_vars = [col for col in df.columns if col not in ['metric', 'value']]
#     df_pivoted = df.pivot_table(
#         index=id_vars,
#         columns='metric',
#         values='value',
#         aggfunc='first'
#     ).reset_index()
    
#     # Get all columns for selection
#     categorical_cols = [col for col in id_vars if df[col].dtype == 'object' or df[col].nunique() < 20]
#     # Remove 'experiment' from categorical_cols if it's there (we'll handle it separately)
#     # if 'experiment' in categorical_cols:
#     #     categorical_cols.remove('experiment')
    
#     numerical_cols = [col for col in id_vars if df[col].dtype in ['float64', 'int64'] and col not in categorical_cols]
#     all_metrics = [col for col in df_pivoted.columns if col not in id_vars]
    
#     # Get unique experiments
#     all_experiments = sorted(df_pivoted['experiment'].unique())
    
#     # Create widgets
#     filter_widgets = {}
#     for col in categorical_cols:
#         unique_vals = sorted(df_pivoted[col].dropna().unique())
#         filter_widgets[col] = widgets.SelectMultiple(
#             options=unique_vals,
#             description=col[:15],
#             layout=widgets.Layout(width='250px'),
#             style={'description_width': 'initial'}
#         )
    
#     # Baseline selection widget
#     baseline_widget = widgets.Dropdown(
#         options=['None'] + all_experiments,
#         value='None',
#         description='Baseline experiment:',
#         layout=widgets.Layout(width='250px'),
#         style={'description_width': 'initial'}
#     )
    
#     # Matching strategy selection
#     matching_strategy_widget = widgets.RadioButtons(
#         options=['Auto (use non-NaN columns)', 'Manual selection'],
#         value='Auto (use non-NaN columns)',
#         description='Matching strategy:',
#         layout=widgets.Layout(width='300px'),
#         style={'description_width': 'initial'}
#     )
    
#     # Manual column selection for matching
#     all_hp_cols = categorical_cols + numerical_cols
#     matching_cols_widget = widgets.SelectMultiple(
#         options=all_hp_cols,
#         value=all_hp_cols[:min(5, len(all_hp_cols))],  # Default to first 5 columns
#         description='Match on:',
#         layout=widgets.Layout(width='300px', height='150px'),
#         style={'description_width': 'initial'},
#         disabled=True  # Initially disabled
#     )
    
#     # Enable/disable manual selection based on strategy
#     def update_matching_cols_enabled(change):
#         matching_cols_widget.disabled = (change['new'] != 'Manual selection')
#     matching_strategy_widget.observe(update_matching_cols_enabled, names='value')
    
#     # Connect lines toggle
#     connect_lines_widget = widgets.Checkbox(
#         value=True,
#         description='Connect to baseline',
#         layout=widgets.Layout(width='200px')
#     )
    
#     # Show baseline only toggle
#     show_baseline_only_widget = widgets.Checkbox(
#         value=False,
#         description='Show only baseline-connected points',
#         layout=widgets.Layout(width='250px')
#     )
    
#     # Show NaN points toggle
#     show_nan_widget = widgets.Checkbox(
#         value=True,
#         description='Show points with NaN values',
#         layout=widgets.Layout(width='250px')
#     )
    
#     # Color by widget
#     color_options = ['None', 'Auto (All columns)'] + categorical_cols + numerical_cols + all_metrics + ['experiment']
#     color_by_widget = widgets.Dropdown(
#         options=color_options,
#         value='experiment',
#         description='Color by:',
#         layout=widgets.Layout(width='250px')
#     )
    
#     # Size by widget
#     size_by_widget = widgets.Dropdown(
#         options=['None'] + numerical_cols + all_metrics,
#         value='None',
#         description='Size by:',
#         layout=widgets.Layout(width='250px')
#     )
    
#     # Opacity slider
#     opacity_slider = widgets.FloatSlider(
#         value=0.7,
#         min=0.1,
#         max=1.0,
#         step=0.05,
#         description='Opacity:',
#         layout=widgets.Layout(width='250px')
#     )
    
#     # Point size slider
#     point_size_slider = widgets.IntSlider(
#         value=8,
#         min=2,
#         max=20,
#         description='Point size:',
#         layout=widgets.Layout(width='250px')
#     )
    
#     # Baseline point style
#     baseline_marker_widget = widgets.Dropdown(
#         options=['circle', 'square', 'diamond', 'cross', 'x', 'star'],
#         value='star',
#         description='Baseline marker:',
#         layout=widgets.Layout(width='250px')
#     )
    
#     # Baseline point size
#     baseline_size_widget = widgets.IntSlider(
#         value=12,
#         min=5,
#         max=25,
#         description='Baseline size:',
#         layout=widgets.Layout(width='250px')
#     )
    
#     # NaN point style
#     nan_marker_widget = widgets.Dropdown(
#         options=['circle', 'square', 'diamond', 'cross', 'x', 'triangle-up', 'triangle-down'],
#         value='diamond',
#         description='NaN marker:',
#         layout=widgets.Layout(width='250px')
#     )
    
#     # Legend toggle
#     legend_toggle = widgets.Checkbox(
#         value=True,
#         description='Show legend',
#         layout=widgets.Layout(width='150px')
#     )
    
#     # Grid toggle
#     grid_toggle = widgets.Checkbox(
#         value=True,
#         description='Show grid',
#         layout=widgets.Layout(width='150px')
#     )
    
#     # Color scale for numerical data
#     color_scale_widget = widgets.Dropdown(
#         options=['Viridis', 'Plasma', 'Inferno', 'Magma', 'Cividis', 'Blues', 'Reds', 'Greens'],
#         value='Viridis',
#         description='Color scale:',
#         layout=widgets.Layout(width='250px')
#     )
    
#     output_widget = Output()
    
#     def get_matching_columns(df_filtered, baseline_name):
#         """Determine which columns to use for matching based on strategy"""
#         if matching_strategy_widget.value == 'Manual selection':
#             # Use manually selected columns
#             matching_cols = list(matching_cols_widget.value)
#             return matching_cols
        
#         # Auto strategy: use columns that don't have NaN in either baseline or non-baseline
#         if baseline_name == 'None' or baseline_name not in df_filtered['experiment'].values:
#             return []
        
#         baseline_df = df_filtered[df_filtered['experiment'] == baseline_name]
#         non_baseline_df = df_filtered[df_filtered['experiment'] != baseline_name]
        
#         valid_cols = []
#         for col in categorical_cols + numerical_cols:
#             if col in df_filtered.columns:
#                 # Check if column has NaN in baseline
#                 baseline_has_nan = baseline_df[col].isna().any()
#                 # Check if column has NaN in non-baseline
#                 non_baseline_has_nan = non_baseline_df[col].isna().any()
                
#                 # Only use column if it has no NaN values in either dataset
#                 if not baseline_has_nan and not non_baseline_has_nan:
#                     valid_cols.append(col)
        
#         return valid_cols
    
#     def find_baseline_matches(df_filtered, baseline_name):
#         """Find matching baseline points using flexible column selection"""
#         if baseline_name == 'None' or baseline_name not in df_filtered['experiment'].values:
#             return [], []
        
#         # Separate baseline and non-baseline data
#         baseline_df = df_filtered[df_filtered['experiment'] == baseline_name].copy()
#         non_baseline_df = df_filtered[df_filtered['experiment'] != baseline_name].copy()
        
#         # Get matching columns based on strategy
#         matching_cols = get_matching_columns(df_filtered, baseline_name)
        
#         if not matching_cols:
#             # If no matching columns found, try to use at least one column
#             available_cols = [col for col in categorical_cols + numerical_cols if col in df_filtered.columns]
#             if available_cols:
#                 matching_cols = [available_cols[0]]
#                 print(f"Warning: No valid matching columns found. Using '{matching_cols[0]}' for matching.")
#             else:
#                 return [], []
        
#         # Create a dictionary for quick lookup of baseline points
#         baseline_dict = {}
#         for idx, row in baseline_df.iterrows():
#             # Create key from matching columns, handling NaN by converting to string
#             hp_key = tuple(str(row[col]) if pd.notna(row[col]) else 'NaN' for col in matching_cols)
#             baseline_dict[hp_key] = row
        
#         # Find matches
#         matches = []
#         matched_baselines = []
        
#         for idx, row in non_baseline_df.iterrows():
#             hp_key = tuple(str(row[col]) if pd.notna(row[col]) else 'NaN' for col in matching_cols)
#             if hp_key in baseline_dict:
#                 baseline_row = baseline_dict[hp_key]
#                 matches.append((row, baseline_row))
#                 matched_baselines.append(baseline_row)
        
#         return matches, matched_baselines
    
#     def update_plot(change=None):
#         with output_widget:
#             clear_output(wait=True)
            
#             # Apply filters
#             filtered_df = df_pivoted.copy()
#             for col, widget in filter_widgets.items():
#                 if widget.value:
#                     filtered_df = filtered_df[filtered_df[col].isin(widget.value)]
            
#             if len(filtered_df) == 0:
#                 print("No data matches the selected filters")
#                 return
            
#             # Display matching strategy info
#             baseline_name = baseline_widget.value
#             if baseline_name != 'None':
#                 matching_cols = get_matching_columns(filtered_df, baseline_name)
#                 if matching_cols:
#                     print(f"🔗 Matching baseline on columns: {', '.join(matching_cols)}")
#                 else:
#                     print("⚠️ No matching columns found! Connections may not work properly.")
            
#             # Filter baseline-only if requested
#             if show_baseline_only_widget.value and baseline_name != 'None':
#                 # Find matches first
#                 matches, _ = find_baseline_matches(filtered_df, baseline_name)
#                 matched_indices = []
#                 for non_base, base in matches:
#                     matched_indices.append(non_base.name)
#                     matched_indices.append(base.name)
#                 filtered_df = filtered_df.loc[matched_indices].drop_duplicates()
                
#                 if len(filtered_df) == 0:
#                     print("No matching points found with baseline")
#                     return
            
#             # Separate points with NaN values if needed
#             show_nan = show_nan_widget.value
#             nan_mask = filtered_df[x_metric].isna() | filtered_df[y_metric].isna()
#             valid_df = filtered_df[~nan_mask].copy() if show_nan else filtered_df.copy()
#             nan_df = filtered_df[nan_mask].copy() if show_nan else pd.DataFrame()
            
#             # Create figure
#             fig = go.Figure()
            
#             # Add baseline connections for valid points
#             connect_lines = connect_lines_widget.value
            
#             if baseline_name != 'None' and connect_lines and len(valid_df) > 0:
#                 # Find matches for lines
#                 matches, _ = find_baseline_matches(valid_df, baseline_name)
                
#                 # Draw lines only for pairs where both points have valid values
#                 lines_drawn = 0
#                 for non_base, base in matches:
#                     # Check if both points have valid (non-NaN) values for the metrics
#                     if (pd.notna(base[x_metric]) and pd.notna(base[y_metric]) and 
#                         pd.notna(non_base[x_metric]) and pd.notna(non_base[y_metric])):
#                         fig.add_trace(go.Scatter(
#                             x=[base[x_metric], non_base[x_metric]],
#                             y=[base[y_metric], non_base[y_metric]],
#                             mode='lines',
#                             line=dict(color='gray', width=1.5, dash='dot'),
#                             showlegend=False,
#                             hoverinfo='none'
#                         ))
#                         lines_drawn += 1
                
#                 if lines_drawn == 0 and len(matches) > 0:
#                     print(f"⚠️ Found {len(matches)} matches but no lines drawn due to NaN values in metrics")
            
#             # Plot valid points (non-NaN)
#             if len(valid_df) > 0:
#                 color_by = color_by_widget.value
                
#                 if color_by == 'None':
#                     temp_fig = px.scatter(
#                         valid_df,
#                         x=x_metric,
#                         y=y_metric,
#                     )
#                     for trace in temp_fig.data:
#                         fig.add_trace(trace)
#                     fig.update_traces(
#                         marker=dict(
#                             opacity=opacity_slider.value,
#                             size=point_size_slider.value
#                         ),
#                         selector=dict(mode='markers')
#                     )
#                 elif color_by == 'Auto (All columns)':
#                     valid_df['color_category'] = valid_df[categorical_cols].astype(str).agg(' | '.join, axis=1)
#                     temp_fig = px.scatter(
#                         valid_df,
#                         x=x_metric,
#                         y=y_metric,
#                         color='color_category',
#                         labels={'color_category': 'Configuration'}
#                     )
#                     for trace in temp_fig.data:
#                         fig.add_trace(trace)
#                     fig.update_traces(
#                         marker=dict(
#                             opacity=opacity_slider.value,
#                             size=point_size_slider.value
#                         ),
#                         selector=dict(mode='markers')
#                     )
#                 else:
#                     is_numerical = (color_by in numerical_cols or color_by in all_metrics) and valid_df[color_by].dtype in ['float64', 'int64']
                    
#                     if is_numerical:
#                         temp_fig = px.scatter(
#                             valid_df,
#                             x=x_metric,
#                             y=y_metric,
#                             color=color_by,
#                             color_continuous_scale=color_scale_widget.value,
#                             labels={color_by: color_by}
#                         )
#                     else:
#                         temp_fig = px.scatter(
#                             valid_df,
#                             x=x_metric,
#                             y=y_metric,
#                             color=color_by,
#                             color_discrete_sequence=px.colors.qualitative.Set1,
#                             labels={color_by: color_by}
#                         )
                    
#                     for trace in temp_fig.data:
#                         fig.add_trace(trace)
                    
#                     fig.update_traces(
#                         marker=dict(
#                             opacity=opacity_slider.value,
#                             size=point_size_slider.value
#                         ),
#                         selector=dict(mode='markers')
#                     )
            
#             # Plot NaN points separately with distinct marker
#             if len(nan_df) > 0 and show_nan:
#                 # Create hover text for NaN points
#                 nan_hover_text = []
#                 for _, row in nan_df.iterrows():
#                     hp_text = '<br>'.join([f'{col}: {row[col]}' for col in categorical_cols if col in nan_df.columns])
#                     nan_hover_text.append(
#                         f"<b>⚠️ INCOMPLETE DATA</b><br>{hp_text}<br>"
#                         f"{x_metric}: {row[x_metric] if pd.notna(row[x_metric]) else 'NaN'}<br>"
#                         f"{y_metric}: {row[y_metric] if pd.notna(row[y_metric]) else 'NaN'}<br>"
#                         f"Experiment: {row['experiment']}"
#                     )
                
#                 fig.add_trace(go.Scatter(
#                     x=nan_df[x_metric] if x_metric in nan_df.columns else [None]*len(nan_df),
#                     y=nan_df[y_metric] if y_metric in nan_df.columns else [None]*len(nan_df),
#                     mode='markers',
#                     name='⚠️ Incomplete data (NaN)',
#                     marker=dict(
#                         symbol=nan_marker_widget.value,
#                         size=point_size_slider.value,
#                         color='gray',
#                         opacity=0.5,
#                         line=dict(color='darkgray', width=1)
#                     ),
#                     text=nan_hover_text,
#                     hovertemplate='%{text}<extra></extra>'
#                 ))
            
#             # Highlight baseline points with different marker (only valid ones)
#             if baseline_name != 'None':
#                 baseline_points = valid_df[valid_df['experiment'] == baseline_name] if len(valid_df) > 0 else pd.DataFrame()
#                 if len(baseline_points) > 0:
#                     # Get matching columns for hover info
#                     matching_cols = get_matching_columns(valid_df, baseline_name)
#                     matching_info = []
#                     for _, row in baseline_points.iterrows():
#                         match_str = '<br>'.join([f'{col}: {row[col]}' for col in matching_cols if col in baseline_points.columns])
#                         matching_info.append(match_str)
                    
#                     fig.add_trace(go.Scatter(
#                         x=baseline_points[x_metric],
#                         y=baseline_points[y_metric],
#                         mode='markers',
#                         name=f'✨ Baseline: {baseline_name}',
#                         marker=dict(
#                             symbol=baseline_marker_widget.value,
#                             size=baseline_size_widget.value,
#                             color='red',
#                             line=dict(color='darkred', width=1)
#                         ),
#                         text=matching_info,
#                         hovertemplate='<b>✨ BASELINE</b><br>%{text}<br>' +
#                                     f'{x_metric}: %{{x:.4f}}<br>{y_metric}: %{{y:.4f}}<extra></extra>'
#                     ))
            
#             # Apply size encoding if selected (only for valid points)
#             if size_by_widget.value != 'None' and len(valid_df) > 0:
#                 size_col = size_by_widget.value
#                 if size_col in valid_df.columns:
#                     size_vals = valid_df[size_col].fillna(valid_df[size_col].median())
#                     min_size, max_size = size_vals.min(), size_vals.max()
#                     if min_size != max_size:
#                         normalized_sizes = 5 + (size_vals - min_size) / (max_size - min_size) * 15
#                     else:
#                         normalized_sizes = [10] * len(valid_df)
                    
#                     # Update only the valid point traces (skip baseline and NaN traces)
#                     for trace in fig.data:
#                         if trace.mode == 'markers' and 'Baseline' not in trace.name and 'Incomplete' not in trace.name:
#                             trace.marker.size = normalized_sizes
#                             trace.marker.sizemode = 'area'
#                             trace.marker.sizeref = 2.*max(normalized_sizes)/(40**2)
            
#             # Add zero lines if requested
#             if signature_flag:
#                 fig.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
#                 fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
            
#             # Update layout based on toggles
#             title = f'{x_metric} vs {y_metric}'
#             if baseline_name != 'None' and connect_lines:
#                 title += f' (connected to baseline: {baseline_name})'
#             if show_nan and len(nan_df) > 0:
#                 title += ' ⚠️ Gray points have NaN values'
            
#             fig.update_layout(
#                 title=title,
#                 plot_bgcolor='white',
#                 xaxis=dict(
#                     title=x_metric,
#                     gridcolor='lightgray',
#                     showgrid=grid_toggle.value,
#                     zeroline=False
#                 ),
#                 yaxis=dict(
#                     title=y_metric,
#                     gridcolor='lightgray',
#                     showgrid=grid_toggle.value,
#                     zeroline=False
#                 ),
#                 hovermode='closest',
#                 height=600,
#                 showlegend=legend_toggle.value
#             )
            
#             fig.show()
    
#     # Create UI layout
#     # Left panel with filters
#     filter_panel = VBox([
#         widgets.HTML("<b>Filters:</b>"),
#         *[HBox([widget]) for widget in filter_widgets.values()]
#     ])
    
#     # Middle panel with baseline controls
#     matching_panel = VBox([
#         widgets.HTML("<b>Matching Strategy:</b>"),
#         matching_strategy_widget,
#         widgets.HTML("<i>Select columns for matching (manual mode):</i>"),
#         matching_cols_widget
#     ])
    
#     baseline_panel = VBox([
#         widgets.HTML("<b>Baseline Settings:</b>"),
#         baseline_widget,
#         connect_lines_widget,
#         show_baseline_only_widget,
#         show_nan_widget,
#         widgets.HTML("<hr>"),
#         widgets.HTML("<b>Marker Styles:</b>"),
#         baseline_marker_widget,
#         baseline_size_widget,
#         nan_marker_widget
#     ])
    
#     # Right panel with styling options
#     style_panel = VBox([
#         widgets.HTML("<b>Styling:</b>"),
#         color_by_widget,
#         size_by_widget,
#         color_scale_widget,
#         opacity_slider,
#         point_size_slider,
#         legend_toggle,
#         grid_toggle
#     ])
    
#     # Main control panel with three columns
#     control_panel = HBox([filter_panel, VBox([matching_panel, baseline_panel]), style_panel])
    
#     # Add reset button
#     reset_button = widgets.Button(description='Reset All', button_style='warning')
    
#     def reset_all(b):
#         for widget in filter_widgets.values():
#             widget.value = []
#         baseline_widget.value = 'None'
#         matching_strategy_widget.value = 'Auto (use non-NaN columns)'
#         matching_cols_widget.value = all_hp_cols[:min(5, len(all_hp_cols))]
#         connect_lines_widget.value = True
#         show_baseline_only_widget.value = False
#         show_nan_widget.value = True
#         color_by_widget.value = 'experiment'
#         size_by_widget.value = 'None'
#         opacity_slider.value = 0.7
#         point_size_slider.value = 8
#         legend_toggle.value = True
#         grid_toggle.value = True
#         color_scale_widget.value = 'Viridis'
#         baseline_marker_widget.value = 'star'
#         baseline_size_widget.value = 12
#         nan_marker_widget.value = 'diamond'
    
#     reset_button.on_click(reset_all)
    
#     # Attach observers
#     for widget in filter_widgets.values():
#         widget.observe(update_plot, names='value')
#     baseline_widget.observe(update_plot, names='value')
#     matching_strategy_widget.observe(update_plot, names='value')
#     matching_cols_widget.observe(update_plot, names='value')
#     connect_lines_widget.observe(update_plot, names='value')
#     show_baseline_only_widget.observe(update_plot, names='value')
#     show_nan_widget.observe(update_plot, names='value')
#     color_by_widget.observe(update_plot, names='value')
#     size_by_widget.observe(update_plot, names='value')
#     opacity_slider.observe(update_plot, names='value')
#     point_size_slider.observe(update_plot, names='value')
#     legend_toggle.observe(update_plot, names='value')
#     grid_toggle.observe(update_plot, names='value')
#     color_scale_widget.observe(update_plot, names='value')
#     baseline_marker_widget.observe(update_plot, names='value')
#     baseline_size_widget.observe(update_plot, names='value')
#     nan_marker_widget.observe(update_plot, names='value')
    
#     # Initial plot
#     update_plot()
    
#     # Display complete interface
#     display(VBox([control_panel, reset_button, output_widget]))
    
#     return filter_widgets, baseline_widget, matching_cols_widget, color_by_widget, size_by_widget

In [28]:
d = compare_prepare('sigmoid_5.2.1', 'baselines_5.2.1', {
#     'dataset': 'cifar10', 
                                                       #  'sketch_method': 'topk', 
                                                        # 'lr': 0.1, 
                                                   #   'smoothing_alpha': 0.9, 
                                                    # 'stabilization_threshold': 1.0
                                                     })

In [44]:
cat[(cat.experiment == 'sigmoid_5.2.1') & (cat.dataset == 'birds')]

,index,dataset,sketch_method,sketch_outputs,smoothing_alpha,stabilization_threshold,subsample,lr,metric,value,std,experiment


In [67]:
create_advanced_interactive_scatter_with_baseline(cat.drop(['index', 'std'], axis=1).fillna('nan'), signature_flag=False)

({'dataset': SelectMultiple(description='dataset', layout=Layout(width='250px'), options=('age_prediction', 'birds', 'cifar10', 'genbase', 'mbd', 'mediamill', 'mnist', 'rt_iot2022', 'yeast'), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_method': SelectMultiple(description='sketch_method', layout=Layout(width='250px'), options=('nan', 'topk'), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_outputs': SelectMultiple(description='sketch_outputs', layout=Layout(width='250px'), options=('1', '10', '101', '12', '13', '14', '19', '2', '20', '25', '27', '3', '4', '5', '50', '6', '7', '75', '9', 'nan'), style=DescriptionStyle(description_width='initial'), value=()),
  'smoothing_alpha': SelectMultiple(description='smoothing_alpha', layout=Layout(width='250px'), options=('0.9', 'nan'), style=DescriptionStyle(description_width='initial'), value=()),
  'stabilization_threshold': SelectMultiple(description='stabilization_t', layout=Layout(width

In [40]:
def create_advanced_interactive_scatter_with_baseline(df, x_metric=None, y_metric=None, signature_flag=False):
    """
    Enhanced version with baseline comparison - with flexible hyperparameter matching.
    Can automatically handle NaN values or manually select matching columns.
    Now with dynamic metric selection widgets.
    
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe in long format
    x_metric, y_metric : str, optional
        Initial metrics to plot (if None, will use first two available metrics)
    signature_flag : bool
        Add zero lines or not
    """
    
    # Pivot the data
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Get all columns for selection
    categorical_cols = [col for col in id_vars if df[col].dtype == 'object' or df[col].nunique() < 20]
    
    numerical_cols = [col for col in id_vars if df[col].dtype in ['float64', 'int64'] and col not in categorical_cols]
    all_metrics = [col for col in df_pivoted.columns if col not in id_vars]
    
    # Set default metrics if not provided
    if x_metric is None and len(all_metrics) > 0:
        x_metric = all_metrics[0]
    if y_metric is None and len(all_metrics) > 1:
        y_metric = all_metrics[1]
    elif y_metric is None and len(all_metrics) > 0:
        y_metric = all_metrics[0]
    
    # Get unique experiments
    all_experiments = sorted(df_pivoted['experiment'].unique())
    
    # Create widgets
    filter_widgets = {}
    for col in categorical_cols:
        unique_vals = sorted(df_pivoted[col].dropna().unique())
        filter_widgets[col] = widgets.SelectMultiple(
            options=unique_vals,
            description=col[:15],
            layout=widgets.Layout(width='250px'),
            style={'description_width': 'initial'}
        )
    
    # Metric selection widgets
    x_metric_widget = widgets.Dropdown(
        options=all_metrics,
        value=x_metric,
        description='X-axis metric:',
        layout=widgets.Layout(width='250px'),
        style={'description_width': 'initial'}
    )
    
    y_metric_widget = widgets.Dropdown(
        options=all_metrics,
        value=y_metric,
        description='Y-axis metric:',
        layout=widgets.Layout(width='250px'),
        style={'description_width': 'initial'}
    )
    
    # Swap axes button
    swap_axes_button = widgets.Button(
        description='⇄ Swap Axes',
        layout=widgets.Layout(width='120px'),
        button_style='primary'
    )
    
    def swap_axes(b):
        current_x = x_metric_widget.value
        current_y = y_metric_widget.value
        x_metric_widget.value = current_y
        y_metric_widget.value = current_x
    
    swap_axes_button.on_click(swap_axes)
    
    # Baseline selection widget
    baseline_widget = widgets.Dropdown(
        options=['None'] + all_experiments,
        value='None',
        description='Baseline experiment:',
        layout=widgets.Layout(width='250px'),
        style={'description_width': 'initial'}
    )
    
    # Matching strategy selection
    matching_strategy_widget = widgets.RadioButtons(
        options=['Auto (use non-NaN columns)', 'Manual selection'],
        value='Auto (use non-NaN columns)',
        description='Matching strategy:',
        layout=widgets.Layout(width='300px'),
        style={'description_width': 'initial'}
    )
    
    # Manual column selection for matching
    all_hp_cols = categorical_cols + numerical_cols
    matching_cols_widget = widgets.SelectMultiple(
        options=all_hp_cols,
        value=all_hp_cols[:min(5, len(all_hp_cols))],  # Default to first 5 columns
        description='Match on:',
        layout=widgets.Layout(width='300px', height='150px'),
        style={'description_width': 'initial'},
        disabled=True  # Initially disabled
    )
    
    # Enable/disable manual selection based on strategy
    def update_matching_cols_enabled(change):
        matching_cols_widget.disabled = (change['new'] != 'Manual selection')
    matching_strategy_widget.observe(update_matching_cols_enabled, names='value')
    
    # Connect lines toggle
    connect_lines_widget = widgets.Checkbox(
        value=True,
        description='Connect to baseline',
        layout=widgets.Layout(width='200px')
    )
    
    # Show baseline only toggle
    show_baseline_only_widget = widgets.Checkbox(
        value=False,
        description='Show only baseline-connected points',
        layout=widgets.Layout(width='250px')
    )
    
    # Show NaN points toggle
    show_nan_widget = widgets.Checkbox(
        value=True,
        description='Show points with NaN values',
        layout=widgets.Layout(width='250px')
    )
    
    # Color by widget
    color_options = ['None', 'Auto (All columns)'] + categorical_cols + numerical_cols + all_metrics + ['experiment']
    color_by_widget = widgets.Dropdown(
        options=color_options,
        value='experiment',
        description='Color by:',
        layout=widgets.Layout(width='250px')
    )
    
    # Size by widget
    size_by_widget = widgets.Dropdown(
        options=['None'] + numerical_cols + all_metrics,
        value='None',
        description='Size by:',
        layout=widgets.Layout(width='250px')
    )
    
    # Opacity slider
    opacity_slider = widgets.FloatSlider(
        value=0.7,
        min=0.1,
        max=1.0,
        step=0.05,
        description='Opacity:',
        layout=widgets.Layout(width='250px')
    )
    
    # Point size slider
    point_size_slider = widgets.IntSlider(
        value=8,
        min=2,
        max=20,
        description='Point size:',
        layout=widgets.Layout(width='250px')
    )
    
    # Baseline point style
    baseline_marker_widget = widgets.Dropdown(
        options=['circle', 'square', 'diamond', 'cross', 'x', 'star'],
        value='star',
        description='Baseline marker:',
        layout=widgets.Layout(width='250px')
    )
    
    # Baseline point size
    baseline_size_widget = widgets.IntSlider(
        value=12,
        min=5,
        max=25,
        description='Baseline size:',
        layout=widgets.Layout(width='250px')
    )
    
    # NaN point style
    nan_marker_widget = widgets.Dropdown(
        options=['circle', 'square', 'diamond', 'cross', 'x', 'triangle-up', 'triangle-down'],
        value='diamond',
        description='NaN marker:',
        layout=widgets.Layout(width='250px')
    )
    
    # Legend toggle
    legend_toggle = widgets.Checkbox(
        value=True,
        description='Show legend',
        layout=widgets.Layout(width='150px')
    )
    
    # Grid toggle
    grid_toggle = widgets.Checkbox(
        value=True,
        description='Show grid',
        layout=widgets.Layout(width='150px')
    )
    
    # Color scale for numerical data
    color_scale_widget = widgets.Dropdown(
        options=['Viridis', 'Plasma', 'Inferno', 'Magma', 'Cividis', 'Blues', 'Reds', 'Greens'],
        value='Viridis',
        description='Color scale:',
        layout=widgets.Layout(width='250px')
    )
    
    output_widget = Output()
    
    def get_matching_columns(df_filtered, baseline_name):
        """Determine which columns to use for matching based on strategy"""
        if matching_strategy_widget.value == 'Manual selection':
            # Use manually selected columns
            matching_cols = list(matching_cols_widget.value)
            return matching_cols
        
        # Auto strategy: use columns that don't have NaN in either baseline or non-baseline
        if baseline_name == 'None' or baseline_name not in df_filtered['experiment'].values:
            return []
        
        baseline_df = df_filtered[df_filtered['experiment'] == baseline_name]
        non_baseline_df = df_filtered[df_filtered['experiment'] != baseline_name]
        
        valid_cols = []
        for col in categorical_cols + numerical_cols:
            if col in df_filtered.columns:
                # Check if column has NaN in baseline
                baseline_has_nan = baseline_df[col].isna().any()
                # Check if column has NaN in non-baseline
                non_baseline_has_nan = non_baseline_df[col].isna().any()
                
                # Only use column if it has no NaN values in either dataset
                if not baseline_has_nan and not non_baseline_has_nan:
                    valid_cols.append(col)
        
        return valid_cols
    
    def find_baseline_matches(df_filtered, baseline_name):
        """Find matching baseline points using flexible column selection"""
        if baseline_name == 'None' or baseline_name not in df_filtered['experiment'].values:
            return [], []
        
        # Separate baseline and non-baseline data
        baseline_df = df_filtered[df_filtered['experiment'] == baseline_name].copy()
        non_baseline_df = df_filtered[df_filtered['experiment'] != baseline_name].copy()
        
        # Get matching columns based on strategy
        matching_cols = get_matching_columns(df_filtered, baseline_name)
        
        if not matching_cols:
            # If no matching columns found, try to use at least one column
            available_cols = [col for col in categorical_cols + numerical_cols if col in df_filtered.columns]
            if available_cols:
                matching_cols = [available_cols[0]]
                print(f"Warning: No valid matching columns found. Using '{matching_cols[0]}' for matching.")
            else:
                return [], []
        
        # Create a dictionary for quick lookup of baseline points
        baseline_dict = {}
        for idx, row in baseline_df.iterrows():
            # Create key from matching columns, handling NaN by converting to string
            hp_key = tuple(str(row[col]) if pd.notna(row[col]) else 'NaN' for col in matching_cols)
            baseline_dict[hp_key] = row
        
        # Find matches
        matches = []
        matched_baselines = []
        
        for idx, row in non_baseline_df.iterrows():
            hp_key = tuple(str(row[col]) if pd.notna(row[col]) else 'NaN' for col in matching_cols)
            if hp_key in baseline_dict:
                baseline_row = baseline_dict[hp_key]
                matches.append((row, baseline_row))
                matched_baselines.append(baseline_row)
        
        return matches, matched_baselines
    
    def update_plot(change=None):
        with output_widget:
            clear_output(wait=True)
            
            # Get current metrics
            current_x_metric = x_metric_widget.value
            current_y_metric = y_metric_widget.value
            
            if current_x_metric is None or current_y_metric is None:
                print("Please select both X and Y metrics")
                return
            
            # Apply filters
            filtered_df = df_pivoted.copy()
            for col, widget in filter_widgets.items():
                if widget.value:
                    filtered_df = filtered_df[filtered_df[col].isin(widget.value)]
            
            if len(filtered_df) == 0:
                print("No data matches the selected filters")
                return
            
            # Display matching strategy info
            baseline_name = baseline_widget.value
            if baseline_name != 'None':
                matching_cols = get_matching_columns(filtered_df, baseline_name)
                if matching_cols:
                    print(f"🔗 Matching baseline on columns: {', '.join(matching_cols)}")
                else:
                    print("⚠️ No matching columns found! Connections may not work properly.")
            
            # Filter baseline-only if requested
            if show_baseline_only_widget.value and baseline_name != 'None':
                # Find matches first
                matches, _ = find_baseline_matches(filtered_df, baseline_name)
                matched_indices = []
                for non_base, base in matches:
                    matched_indices.append(non_base.name)
                    matched_indices.append(base.name)
                filtered_df = filtered_df.loc[matched_indices].drop_duplicates()
                
                if len(filtered_df) == 0:
                    print("No matching points found with baseline")
                    return
            
            # Separate points with NaN values if needed
            show_nan = show_nan_widget.value
            nan_mask = filtered_df[current_x_metric].isna() | filtered_df[current_y_metric].isna()
            valid_df = filtered_df[~nan_mask].copy() if show_nan else filtered_df.copy()
            nan_df = filtered_df[nan_mask].copy() if show_nan else pd.DataFrame()
            
            # Create figure
            fig = go.Figure()
            
            # Add baseline connections for valid points
            connect_lines = connect_lines_widget.value
            
            if baseline_name != 'None' and connect_lines and len(valid_df) > 0:
                # Find matches for lines
                matches, _ = find_baseline_matches(valid_df, baseline_name)
                
                # Draw lines only for pairs where both points have valid values
                lines_drawn = 0
                for non_base, base in matches:
                    # Check if both points have valid (non-NaN) values for the metrics
                    if (pd.notna(base[current_x_metric]) and pd.notna(base[current_y_metric]) and 
                        pd.notna(non_base[current_x_metric]) and pd.notna(non_base[current_y_metric])):
                        fig.add_trace(go.Scatter(
                            x=[base[current_x_metric], non_base[current_x_metric]],
                            y=[base[current_y_metric], non_base[current_y_metric]],
                            mode='lines',
                            line=dict(color='gray', width=1.5, dash='dot'),
                            showlegend=False,
                            hoverinfo='none'
                        ))
                        lines_drawn += 1
                
                if lines_drawn == 0 and len(matches) > 0:
                    print(f"⚠️ Found {len(matches)} matches but no lines drawn due to NaN values in metrics")
            
            # Plot valid points (non-NaN)
            if len(valid_df) > 0:
                color_by = color_by_widget.value
                
                if color_by == 'None':
                    temp_fig = px.scatter(
                        valid_df,
                        x=current_x_metric,
                        y=current_y_metric,
                    )
                    for trace in temp_fig.data:
                        fig.add_trace(trace)
                    fig.update_traces(
                        marker=dict(
                            opacity=opacity_slider.value,
                            size=point_size_slider.value
                        ),
                        selector=dict(mode='markers')
                    )
                elif color_by == 'Auto (All columns)':
                    valid_df['color_category'] = valid_df[categorical_cols].astype(str).agg(' | '.join, axis=1)
                    temp_fig = px.scatter(
                        valid_df,
                        x=current_x_metric,
                        y=current_y_metric,
                        color='color_category',
                        labels={'color_category': 'Configuration'}
                    )
                    for trace in temp_fig.data:
                        fig.add_trace(trace)
                    fig.update_traces(
                        marker=dict(
                            opacity=opacity_slider.value,
                            size=point_size_slider.value
                        ),
                        selector=dict(mode='markers')
                    )
                else:
                    is_numerical = (color_by in numerical_cols or color_by in all_metrics) and valid_df[color_by].dtype in ['float64', 'int64']
                    
                    if is_numerical:
                        temp_fig = px.scatter(
                            valid_df,
                            x=current_x_metric,
                            y=current_y_metric,
                            color=color_by,
                            color_continuous_scale=color_scale_widget.value,
                            labels={color_by: color_by}
                        )
                    else:
                        temp_fig = px.scatter(
                            valid_df,
                            x=current_x_metric,
                            y=current_y_metric,
                            color=color_by,
                            color_discrete_sequence=px.colors.qualitative.Set1,
                            labels={color_by: color_by}
                        )
                    
                    for trace in temp_fig.data:
                        fig.add_trace(trace)
                    
                    fig.update_traces(
                        marker=dict(
                            opacity=opacity_slider.value,
                            size=point_size_slider.value
                        ),
                        selector=dict(mode='markers')
                    )
            
            # Plot NaN points separately with distinct marker
            if len(nan_df) > 0 and show_nan:
                # Create hover text for NaN points
                nan_hover_text = []
                for _, row in nan_df.iterrows():
                    hp_text = '<br>'.join([f'{col}: {row[col]}' for col in categorical_cols if col in nan_df.columns])
                    nan_hover_text.append(
                        f"<b>⚠️ INCOMPLETE DATA</b><br>{hp_text}<br>"
                        f"{current_x_metric}: {row[current_x_metric] if pd.notna(row[current_x_metric]) else 'NaN'}<br>"
                        f"{current_y_metric}: {row[current_y_metric] if pd.notna(row[current_y_metric]) else 'NaN'}<br>"
                        f"Experiment: {row['experiment']}"
                    )
                
                fig.add_trace(go.Scatter(
                    x=nan_df[current_x_metric] if current_x_metric in nan_df.columns else [None]*len(nan_df),
                    y=nan_df[current_y_metric] if current_y_metric in nan_df.columns else [None]*len(nan_df),
                    mode='markers',
                    name='⚠️ Incomplete data (NaN)',
                    marker=dict(
                        symbol=nan_marker_widget.value,
                        size=point_size_slider.value,
                        color='gray',
                        opacity=0.5,
                        line=dict(color='darkgray', width=1)
                    ),
                    text=nan_hover_text,
                    hovertemplate='%{text}<extra></extra>'
                ))
            
            # Highlight baseline points with different marker (only valid ones)
            if baseline_name != 'None':
                baseline_points = valid_df[valid_df['experiment'] == baseline_name] if len(valid_df) > 0 else pd.DataFrame()
                if len(baseline_points) > 0:
                    # Get matching columns for hover info
                    matching_cols = get_matching_columns(valid_df, baseline_name)
                    matching_info = []
                    for _, row in baseline_points.iterrows():
                        match_str = '<br>'.join([f'{col}: {row[col]}' for col in matching_cols if col in baseline_points.columns])
                        matching_info.append(match_str)
                    
                    fig.add_trace(go.Scatter(
                        x=baseline_points[current_x_metric],
                        y=baseline_points[current_y_metric],
                        mode='markers',
                        name=f'✨ Baseline: {baseline_name}',
                        marker=dict(
                            symbol=baseline_marker_widget.value,
                            size=baseline_size_widget.value,
                            color='red',
                            line=dict(color='darkred', width=1)
                        ),
                        text=matching_info,
                        hovertemplate='<b>✨ BASELINE</b><br>%{text}<br>' +
                                    f'{current_x_metric}: %{{x:.4f}}<br>{current_y_metric}: %{{y:.4f}}<extra></extra>'
                    ))
            
            # Apply size encoding if selected (only for valid points)
            if size_by_widget.value != 'None' and len(valid_df) > 0:
                size_col = size_by_widget.value
                if size_col in valid_df.columns:
                    size_vals = valid_df[size_col].fillna(valid_df[size_col].median())
                    min_size, max_size = size_vals.min(), size_vals.max()
                    if min_size != max_size:
                        normalized_sizes = 5 + (size_vals - min_size) / (max_size - min_size) * 15
                    else:
                        normalized_sizes = [10] * len(valid_df)
                    
                    # Update only the valid point traces (skip baseline and NaN traces)
                    for trace in fig.data:
                        if trace.mode == 'markers' and 'Baseline' not in trace.name and 'Incomplete' not in trace.name:
                            trace.marker.size = normalized_sizes
                            trace.marker.sizemode = 'area'
                            trace.marker.sizeref = 2.*max(normalized_sizes)/(40**2)
            
            # Add zero lines if requested
            if signature_flag:
                fig.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
                fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
            
            # Update layout based on toggles
            title = f'{current_x_metric} vs {current_y_metric}'
            if baseline_name != 'None' and connect_lines:
                title += f' (connected to baseline: {baseline_name})'
            if show_nan and len(nan_df) > 0:
                title += ' ⚠️ Gray points have NaN values'
            
            fig.update_layout(
                title=title,
                plot_bgcolor='white',
                xaxis=dict(
                    title=current_x_metric,
                    gridcolor='lightgray',
                    showgrid=grid_toggle.value,
                    zeroline=False
                ),
                yaxis=dict(
                    title=current_y_metric,
                    gridcolor='lightgray',
                    showgrid=grid_toggle.value,
                    zeroline=False
                ),
                hovermode='closest',
                height=600,
                showlegend=legend_toggle.value
            )
            
            fig.show()
    
    # Create UI layout
    # Top panel for metric selection
    metric_panel = HBox([
        x_metric_widget,
        y_metric_widget,
        swap_axes_button
    ])
    
    # Left panel with filters
    filter_panel = VBox([
        widgets.HTML("<b>Filters:</b>"),
        *[HBox([widget]) for widget in filter_widgets.values()]
    ])
    
    # Middle panel with baseline controls
    matching_panel = VBox([
        widgets.HTML("<b>Matching Strategy:</b>"),
        matching_strategy_widget,
        widgets.HTML("<i>Select columns for matching (manual mode):</i>"),
        matching_cols_widget
    ])
    
    baseline_panel = VBox([
        widgets.HTML("<b>Baseline Settings:</b>"),
        baseline_widget,
        connect_lines_widget,
        show_baseline_only_widget,
        show_nan_widget,
        widgets.HTML("<hr>"),
        widgets.HTML("<b>Marker Styles:</b>"),
        baseline_marker_widget,
        baseline_size_widget,
        nan_marker_widget
    ])
    
    # Right panel with styling options
    style_panel = VBox([
        widgets.HTML("<b>Styling:</b>"),
        color_by_widget,
        size_by_widget,
        color_scale_widget,
        opacity_slider,
        point_size_slider,
        legend_toggle,
        grid_toggle
    ])
    
    # Main control panel with three columns
    control_panel = VBox([
        metric_panel,
        widgets.HTML("<hr>"),
        HBox([filter_panel, VBox([matching_panel, baseline_panel]), style_panel])
    ])
    
    # Add reset button
    reset_button = widgets.Button(description='Reset All', button_style='warning')
    
    def reset_all(b):
        for widget in filter_widgets.values():
            widget.value = []
        # Reset metrics to first two if available
        if len(all_metrics) > 0:
            x_metric_widget.value = all_metrics[0]
        if len(all_metrics) > 1:
            y_metric_widget.value = all_metrics[1]
        elif len(all_metrics) > 0:
            y_metric_widget.value = all_metrics[0]
        baseline_widget.value = 'None'
        matching_strategy_widget.value = 'Auto (use non-NaN columns)'
        matching_cols_widget.value = all_hp_cols[:min(5, len(all_hp_cols))]
        connect_lines_widget.value = True
        show_baseline_only_widget.value = False
        show_nan_widget.value = True
        color_by_widget.value = 'experiment'
        size_by_widget.value = 'None'
        opacity_slider.value = 0.7
        point_size_slider.value = 8
        legend_toggle.value = True
        grid_toggle.value = True
        color_scale_widget.value = 'Viridis'
        baseline_marker_widget.value = 'star'
        baseline_size_widget.value = 12
        nan_marker_widget.value = 'diamond'
    
    reset_button.on_click(reset_all)
    
    # Attach observers
    for widget in filter_widgets.values():
        widget.observe(update_plot, names='value')
    x_metric_widget.observe(update_plot, names='value')
    y_metric_widget.observe(update_plot, names='value')
    baseline_widget.observe(update_plot, names='value')
    matching_strategy_widget.observe(update_plot, names='value')
    matching_cols_widget.observe(update_plot, names='value')
    connect_lines_widget.observe(update_plot, names='value')
    show_baseline_only_widget.observe(update_plot, names='value')
    show_nan_widget.observe(update_plot, names='value')
    color_by_widget.observe(update_plot, names='value')
    size_by_widget.observe(update_plot, names='value')
    opacity_slider.observe(update_plot, names='value')
    point_size_slider.observe(update_plot, names='value')
    legend_toggle.observe(update_plot, names='value')
    grid_toggle.observe(update_plot, names='value')
    color_scale_widget.observe(update_plot, names='value')
    baseline_marker_widget.observe(update_plot, names='value')
    baseline_size_widget.observe(update_plot, names='value')
    nan_marker_widget.observe(update_plot, names='value')
    
    # Initial plot
    update_plot()
    
    # Display complete interface
    display(VBox([control_panel, reset_button, output_widget]))
    
    return (filter_widgets, x_metric_widget, y_metric_widget, baseline_widget, 
            matching_cols_widget, color_by_widget, size_by_widget)

In [42]:
cat[cat.dataset == 'cifar10']

,index,dataset,sketch_method,sketch_outputs,smoothing_alpha,stabilization_threshold,subsample,lr,metric,value,std,experiment
4500,4500,cifar10,topk,1,0.9,0.5,0.05,0.005,accuracy,0.434667,0.013512,sigmoid_5.2.1
4501,4501,cifar10,topk,1,0.9,0.5,0.05,0.005,adaptive_threshold_applied,NaN,NaN,sigmoid_5.2.1
4502,4502,cifar10,topk,1,0.9,0.5,0.05,0.005,adaptive_threshold_fallback,NaN,NaN,sigmoid_5.2.1
4503,4503,cifar10,topk,1,0.9,0.5,0.05,0.005,adaptive_threshold_inference_time,NaN,NaN,sigmoid_5.2.1
4504,4504,cifar10,topk,1,0.9,0.5,0.05,0.005,bce_loss,NaN,NaN,sigmoid_5.2.1
...,...,...,...,...,...,...,...,...,...,...,...,...
208,208,cifar10,NaN,NaN,NaN,NaN,0.05,0.1,hamming_loss,0.761083,0.006748,xgboost_5.2.1
224,224,cifar10,NaN,NaN,NaN,NaN,0.5,0.005,hamming_loss,0.856500,0.006976,xgboost_5.2.1
240,240,cifar10,NaN,NaN,NaN,NaN,0.5,0.1,hamming_loss,0.759917,0.006855,xgboost_5.2.1
256,256,cifar10,NaN,NaN,NaN,NaN,0.75,0.005,hamming_loss,0.857583,0.008022,xgboost_5.2.1


In [ ]:
create_advanced_interactive_scatter_with_baseline(cat.drop(['std', 'index'], axis=1).fillna('nan'),)

({'dataset': SelectMultiple(description='dataset', layout=Layout(width='250px'), options=('age_prediction', 'birds', 'cifar10', 'genbase', 'mbd', 'mediamill', 'mnist', 'rt_iot2022', 'yeast'), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_method': SelectMultiple(description='sketch_method', layout=Layout(width='250px'), options=('nan', 'topk'), style=DescriptionStyle(description_width='initial'), value=()),
  'sketch_outputs': SelectMultiple(description='sketch_outputs', layout=Layout(width='250px'), options=('1', '10', '101', '12', '13', '14', '19', '2', '20', '25', '27', '3', '4', '5', '50', '6', '7', '75', '9', 'nan'), style=DescriptionStyle(description_width='initial'), value=()),
  'smoothing_alpha': SelectMultiple(description='smoothing_alpha', layout=Layout(width='250px'), options=('0.9', 'nan'), style=DescriptionStyle(description_width='initial'), value=()),
  'stabilization_threshold': SelectMultiple(description='stabilization_t', layout=Layout(width

In [37]:
def create_advanced_interactive_scatter_with_baseline(df, x_metric, y_metric, signature_flag=False):
    """
    Enhanced version with baseline comparison - INCLUDES records with NaN values.
    NaN values are shown but not connected to baseline.
    """
    
    # Pivot the data
    id_vars = [col for col in df.columns if col not in ['metric', 'value']]
    df_pivoted = df.pivot_table(
        index=id_vars,
        columns='metric',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Get all columns for selection
    categorical_cols = [col for col in id_vars if df[col].dtype == 'object' or df[col].nunique() < 20]
    # Remove 'experiment' from categorical_cols if it's there (we'll handle it separately)
    if 'experiment' in categorical_cols:
        categorical_cols.remove('experiment')
    
    numerical_cols = [col for col in id_vars if df[col].dtype in ['float64', 'int64'] and col not in categorical_cols]
    all_metrics = [col for col in df_pivoted.columns if col not in id_vars]
    
    # Get unique experiments
    all_experiments = sorted(df_pivoted['experiment'].unique())
    
    # Create widgets
    filter_widgets = {}
    for col in categorical_cols:
        unique_vals = sorted(df_pivoted[col].dropna().unique())
        filter_widgets[col] = widgets.SelectMultiple(
            options=unique_vals,
            description=col[:15],
            layout=widgets.Layout(width='250px'),
            style={'description_width': 'initial'}
        )
    
    # Baseline selection widget
    baseline_widget = widgets.Dropdown(
        options=['None'] + all_experiments,
        value='None',
        description='Baseline experiment:',
        layout=widgets.Layout(width='250px'),
        style={'description_width': 'initial'}
    )
    
    # Connect lines toggle
    connect_lines_widget = widgets.Checkbox(
        value=True,
        description='Connect to baseline',
        layout=widgets.Layout(width='200px')
    )
    
    # Show baseline only toggle
    show_baseline_only_widget = widgets.Checkbox(
        value=False,
        description='Show only baseline-connected points',
        layout=widgets.Layout(width='250px')
    )
    
    # Show NaN points toggle
    show_nan_widget = widgets.Checkbox(
        value=True,
        description='Show points with NaN values',
        layout=widgets.Layout(width='250px')
    )
    
    # Color by widget
    color_options = ['None', 'Auto (All columns)'] + categorical_cols + numerical_cols + all_metrics + ['experiment']
    color_by_widget = widgets.Dropdown(
        options=color_options,
        value='experiment',
        description='Color by:',
        layout=widgets.Layout(width='250px')
    )
    
    # Size by widget
    size_by_widget = widgets.Dropdown(
        options=['None'] + numerical_cols + all_metrics,
        value='None',
        description='Size by:',
        layout=widgets.Layout(width='250px')
    )
    
    # Opacity slider
    opacity_slider = widgets.FloatSlider(
        value=0.7,
        min=0.1,
        max=1.0,
        step=0.05,
        description='Opacity:',
        layout=widgets.Layout(width='250px')
    )
    
    # Point size slider
    point_size_slider = widgets.IntSlider(
        value=8,
        min=2,
        max=20,
        description='Point size:',
        layout=widgets.Layout(width='250px')
    )
    
    # Baseline point style
    baseline_marker_widget = widgets.Dropdown(
        options=['circle', 'square', 'diamond', 'cross', 'x', 'star'],
        value='star',
        description='Baseline marker:',
        layout=widgets.Layout(width='250px')
    )
    
    # Baseline point size
    baseline_size_widget = widgets.IntSlider(
        value=12,
        min=5,
        max=25,
        description='Baseline size:',
        layout=widgets.Layout(width='250px')
    )
    
    # NaN point style
    nan_marker_widget = widgets.Dropdown(
        options=['circle', 'square', 'diamond', 'cross', 'x', 'triangle-up', 'triangle-down'],
        value='diamond',
        description='NaN marker:',
        layout=widgets.Layout(width='250px')
    )
    
    # Legend toggle
    legend_toggle = widgets.Checkbox(
        value=True,
        description='Show legend',
        layout=widgets.Layout(width='150px')
    )
    
    # Grid toggle
    grid_toggle = widgets.Checkbox(
        value=True,
        description='Show grid',
        layout=widgets.Layout(width='150px')
    )
    
    # Color scale for numerical data
    color_scale_widget = widgets.Dropdown(
        options=['Viridis', 'Plasma', 'Inferno', 'Magma', 'Cividis', 'Blues', 'Reds', 'Greens'],
        value='Viridis',
        description='Color scale:',
        layout=widgets.Layout(width='250px')
    )
    
    output_widget = Output()
    
    def find_baseline_matches(df_filtered, baseline_name):
        """Find matching baseline points for each non-baseline point"""
        if baseline_name == 'None' or baseline_name not in df_filtered['experiment'].values:
            return [], []
        
        # Separate baseline and non-baseline data
        baseline_df = df_filtered[df_filtered['experiment'] == baseline_name].copy()
        non_baseline_df = df_filtered[df_filtered['experiment'] != baseline_name].copy()
        
        # Identify hyperparameter columns (all categorical_cols)
        hp_cols = [col for col in categorical_cols if col in df_filtered.columns]
        
        if not hp_cols:
            return [], []
        
        # Create a dictionary for quick lookup of baseline points by hyperparameters
        baseline_dict = {}
        for _, row in baseline_df.iterrows():
            hp_key = tuple(row[hp_cols].values)
            baseline_dict[hp_key] = row
        
        # Find matches
        matches = []
        matched_baselines = []
        
        for _, row in non_baseline_df.iterrows():
            hp_key = tuple(row[hp_cols].values)
            if hp_key in baseline_dict:
                baseline_row = baseline_dict[hp_key]
                # Check if baseline has non-NaN values for the metrics
                # But still include the match even if NaN (just won't draw line)
                matches.append((row, baseline_row))
                matched_baselines.append(baseline_row)
        
        return matches, matched_baselines
    
    def update_plot(change=None):
        with output_widget:
            clear_output(wait=True)
            
            # Apply filters
            filtered_df = df_pivoted.copy()
            for col, widget in filter_widgets.items():
                if widget.value:
                    filtered_df = filtered_df[filtered_df[col].isin(widget.value)]
            
            if len(filtered_df) == 0:
                print("No data matches the selected filters")
                return
            
            # Filter baseline-only if requested
            baseline_name = baseline_widget.value
            if show_baseline_only_widget.value and baseline_name != 'None':
                # Find matches first
                matches, _ = find_baseline_matches(filtered_df, baseline_name)
                matched_indices = []
                for non_base, base in matches:
                    matched_indices.append(non_base.name)
                    matched_indices.append(base.name)
                filtered_df = filtered_df.loc[matched_indices].drop_duplicates()
                
                if len(filtered_df) == 0:
                    print("No matching points found with baseline")
                    return
            
            # Separate points with NaN values if needed
            show_nan = show_nan_widget.value
            nan_mask = filtered_df[x_metric].isna() | filtered_df[y_metric].isna()
            valid_df = filtered_df[~nan_mask].copy() if show_nan else filtered_df.copy()
            nan_df = filtered_df[nan_mask].copy() if show_nan else pd.DataFrame()
            
            # Create figure
            fig = go.Figure()
            
            # Add baseline connections for valid points
            baseline_name = baseline_widget.value
            connect_lines = connect_lines_widget.value
            
            if baseline_name != 'None' and connect_lines and len(valid_df) > 0:
                # Find matches for lines
                matches, _ = find_baseline_matches(valid_df, baseline_name)
                
                # Draw lines only for pairs where both points have valid values
                for non_base, base in matches:
                    # Check if both points have valid (non-NaN) values for the metrics
                    if (pd.notna(base[x_metric]) and pd.notna(base[y_metric]) and 
                        pd.notna(non_base[x_metric]) and pd.notna(non_base[y_metric])):
                        fig.add_trace(go.Scatter(
                            x=[base[x_metric], non_base[x_metric]],
                            y=[base[y_metric], non_base[y_metric]],
                            mode='lines',
                            line=dict(color='gray', width=1.5, dash='dot'),
                            showlegend=False,
                            hoverinfo='none'
                        ))
            
            # Plot valid points (non-NaN)
            if len(valid_df) > 0:
                color_by = color_by_widget.value
                
                if color_by == 'None':
                    temp_fig = px.scatter(
                        valid_df,
                        x=x_metric,
                        y=y_metric,
                    )
                    for trace in temp_fig.data:
                        fig.add_trace(trace)
                    fig.update_traces(
                        marker=dict(
                            opacity=opacity_slider.value,
                            size=point_size_slider.value
                        ),
                        selector=dict(mode='markers')
                    )
                elif color_by == 'Auto (All columns)':
                    valid_df['color_category'] = valid_df[categorical_cols].astype(str).agg(' | '.join, axis=1)
                    temp_fig = px.scatter(
                        valid_df,
                        x=x_metric,
                        y=y_metric,
                        color='color_category',
                        labels={'color_category': 'Configuration'}
                    )
                    for trace in temp_fig.data:
                        fig.add_trace(trace)
                    fig.update_traces(
                        marker=dict(
                            opacity=opacity_slider.value,
                            size=point_size_slider.value
                        ),
                        selector=dict(mode='markers')
                    )
                else:
                    is_numerical = (color_by in numerical_cols or color_by in all_metrics) and valid_df[color_by].dtype in ['float64', 'int64']
                    
                    if is_numerical:
                        temp_fig = px.scatter(
                            valid_df,
                            x=x_metric,
                            y=y_metric,
                            color=color_by,
                            color_continuous_scale=color_scale_widget.value,
                            labels={color_by: color_by}
                        )
                    else:
                        temp_fig = px.scatter(
                            valid_df,
                            x=x_metric,
                            y=y_metric,
                            color=color_by,
                            color_discrete_sequence=px.colors.qualitative.Set1,
                            labels={color_by: color_by}
                        )
                    
                    for trace in temp_fig.data:
                        fig.add_trace(trace)
                    
                    fig.update_traces(
                        marker=dict(
                            opacity=opacity_slider.value,
                            size=point_size_slider.value
                        ),
                        selector=dict(mode='markers')
                    )
            
            # Plot NaN points separately with distinct marker
            if len(nan_df) > 0 and show_nan:
                # Create hover text for NaN points
                nan_hover_text = []
                for _, row in nan_df.iterrows():
                    hp_text = '<br>'.join([f'{col}: {row[col]}' for col in categorical_cols if col in nan_df.columns])
                    nan_hover_text.append(
                        f"<b>⚠️ INCOMPLETE DATA</b><br>{hp_text}<br>"
                        f"{x_metric}: {row[x_metric] if pd.notna(row[x_metric]) else 'NaN'}<br>"
                        f"{y_metric}: {row[y_metric] if pd.notna(row[y_metric]) else 'NaN'}<br>"
                        f"Experiment: {row['experiment']}"
                    )
                
                fig.add_trace(go.Scatter(
                    x=nan_df[x_metric] if x_metric in nan_df.columns else [None]*len(nan_df),
                    y=nan_df[y_metric] if y_metric in nan_df.columns else [None]*len(nan_df),
                    mode='markers',
                    name='⚠️ Incomplete data (NaN)',
                    marker=dict(
                        symbol=nan_marker_widget.value,
                        size=point_size_slider.value,
                        color='gray',
                        opacity=0.5,
                        line=dict(color='darkgray', width=1)
                    ),
                    text=nan_hover_text,
                    hovertemplate='%{text}<extra></extra>'
                ))
            
            # Highlight baseline points with different marker (only valid ones)
            if baseline_name != 'None':
                baseline_points = valid_df[valid_df['experiment'] == baseline_name] if len(valid_df) > 0 else pd.DataFrame()
                if len(baseline_points) > 0:
                    fig.add_trace(go.Scatter(
                        x=baseline_points[x_metric],
                        y=baseline_points[y_metric],
                        mode='markers',
                        name=f'✨ Baseline: {baseline_name}',
                        marker=dict(
                            symbol=baseline_marker_widget.value,
                            size=baseline_size_widget.value,
                            color='red',
                            line=dict(color='darkred', width=1)
                        ),
                        text=baseline_points[categorical_cols].astype(str).agg(' | '.join, axis=1),
                        hovertemplate='<b>✨ BASELINE</b><br>' + 
                                    '<br>'.join([f'{col}: %{{text.split(" | ")[{i}]}}' for i, col in enumerate(categorical_cols) if col in baseline_points.columns]) +
                                    f'<br>{x_metric}: %{{x:.4f}}<br>{y_metric}: %{{y:.4f}}<extra></extra>'
                    ))
            
            # Apply size encoding if selected (only for valid points)
            if size_by_widget.value != 'None' and len(valid_df) > 0:
                size_col = size_by_widget.value
                if size_col in valid_df.columns:
                    size_vals = valid_df[size_col].fillna(valid_df[size_col].median())
                    min_size, max_size = size_vals.min(), size_vals.max()
                    if min_size != max_size:
                        normalized_sizes = 5 + (size_vals - min_size) / (max_size - min_size) * 15
                    else:
                        normalized_sizes = [10] * len(valid_df)
                    
                    # Update only the valid point traces (skip baseline and NaN traces)
                    for trace in fig.data:
                        if trace.mode == 'markers' and 'Baseline' not in trace.name and 'Incomplete' not in trace.name:
                            trace.marker.size = normalized_sizes
                            trace.marker.sizemode = 'area'
                            trace.marker.sizeref = 2.*max(normalized_sizes)/(40**2)
            
            # Add zero lines if requested
            if signature_flag:
                fig.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
                fig.add_hline(y=0, line_width=1.5, line_dash="dash", line_color="gray", opacity=0.7)
            
            # Update layout based on toggles
            title = f'{x_metric} vs {y_metric}'
            if baseline_name != 'None' and connect_lines:
                title += f' (connected to baseline: {baseline_name})'
            if show_nan and len(nan_df) > 0:
                title += ' ⚠️ Gray points have NaN values'
            
            fig.update_layout(
                title=title,
                plot_bgcolor='white',
                xaxis=dict(
                    title=x_metric,
                    gridcolor='lightgray',
                    showgrid=grid_toggle.value,
                    zeroline=False
                ),
                yaxis=dict(
                    title=y_metric,
                    gridcolor='lightgray',
                    showgrid=grid_toggle.value,
                    zeroline=False
                ),
                hovermode='closest',
                height=600,
                showlegend=legend_toggle.value
            )
            
            fig.show()
    
    # Create UI layout
    # Left panel with filters
    filter_panel = VBox([
        widgets.HTML("<b>Filters:</b>"),
        *[HBox([widget]) for widget in filter_widgets.values()]
    ])
    
    # Middle panel with baseline controls
    baseline_panel = VBox([
        widgets.HTML("<b>Baseline Settings:</b>"),
        baseline_widget,
        connect_lines_widget,
        show_baseline_only_widget,
        show_nan_widget,
        baseline_marker_widget,
        baseline_size_widget,
        nan_marker_widget
    ])
    
    # Right panel with styling options
    style_panel = VBox([
        widgets.HTML("<b>Styling:</b>"),
        color_by_widget,
        size_by_widget,
        color_scale_widget,
        opacity_slider,
        point_size_slider,
        legend_toggle,
        grid_toggle
    ])
    
    # Main control panel
    control_panel = HBox([filter_panel, baseline_panel, style_panel])
    
    # Add reset button
    reset_button = widgets.Button(description='Reset All', button_style='warning')
    
    def reset_all(b):
        for widget in filter_widgets.values():
            widget.value = []
        baseline_widget.value = 'None'
        connect_lines_widget.value = True
        show_baseline_only_widget.value = False
        show_nan_widget.value = True
        color_by_widget.value = 'experiment'
        size_by_widget.value = 'None'
        opacity_slider.value = 0.7
        point_size_slider.value = 8
        legend_toggle.value = True
        grid_toggle.value = True
        color_scale_widget.value = 'Viridis'
        baseline_marker_widget.value = 'star'
        baseline_size_widget.value = 12
        nan_marker_widget.value = 'diamond'
    
    reset_button.on_click(reset_all)
    
    # Attach observers
    for widget in filter_widgets.values():
        widget.observe(update_plot, names='value')
    baseline_widget.observe(update_plot, names='value')
    connect_lines_widget.observe(update_plot, names='value')
    show_baseline_only_widget.observe(update_plot, names='value')
    show_nan_widget.observe(update_plot, names='value')
    color_by_widget.observe(update_plot, names='value')
    size_by_widget.observe(update_plot, names='value')
    opacity_slider.observe(update_plot, names='value')
    point_size_slider.observe(update_plot, names='value')
    legend_toggle.observe(update_plot, names='value')
    grid_toggle.observe(update_plot, names='value')
    color_scale_widget.observe(update_plot, names='value')
    baseline_marker_widget.observe(update_plot, names='value')
    baseline_size_widget.observe(update_plot, names='value')
    nan_marker_widget.observe(update_plot, names='value')
    
    # Initial plot
    update_plot()
    
    # Display complete interface
    display(VBox([control_panel, reset_button, output_widget]))
    
    return filter_widgets, baseline_widget, color_by_widget, size_by_widget

In [32]:
create_interactive_metric_scatter(compare_prepare('sigmoid_5.2', 'baselines_5.2', {}), 'inference_time', 'f1', signature_flag=False)

KeyError: 'sigmoid_5.2'

In [35]:
import seaborn as sns
d = compare_prepare('sigmoid_5.2', 'baselines_5.2', {})
sns.scatterplot(filter_df(d, {'lr': 0.1, 'sketch_outputs': 2}), )

KeyError: 'sigmoid_5.2'

In [28]:
create_interactive_metric_scatter(d, 'inference_time', 'accuracy', signature_flag=True)

NameError: name 'create_interactive_metric_scatter' is not defined

## Analogues

In [235]:
analogues_names = ['analogues_3.1', 'analogues_lgbm_3.1']

analogues = dict(zip(analogues_names, (get_all_runs_data([analogues_name]) for analogues_name in analogues_names)))

Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: analogues_3.1


Processing experiments:   0%|          | 0/1 [00:00<?, ?it/s]

Processing experiment: analogues_lgbm_3.1


Processing experiments: 100%|██████████| 1/1 [00:00<00:00, 12.12it/s]


In [236]:
def filter_data(data):
    after_exclusion_by_name = [
        col for col in data.columns if col not in EXCLUDE
    ]
    statistics = ('mean', 'max', 'min', 'std', 'median')
    after_exclusion_agg = [
        col for col in after_exclusion_by_name if 'leaves' in col or 'nodes' in col or
        'tree' in col or
        not any(statistic in col for statistic in statistics) and not 'metric_fold' in col
    ]
    filtered_data = data[after_exclusion_agg 
                        #  + ['metric_std_num_trees', 'metric_mean_num_trees',]
                         ]
    filtered_pivot = filtered_data.rename(columns={col: col.removeprefix('param_') for col in filtered_data.columns})
    return filtered_pivot

def melt_metrics(df):
    # Identify metric columns
    metric_cols = [col for col in df.columns if col.startswith('metric_')]
    
    # Identify ID columns (all non-metric columns)
    id_cols = [col for col in df.columns if not col.startswith('metric_')]
    
    # Melt using pandas melt (more control)
    melted_df = df.melt(
        id_vars=id_cols,
        value_vars=metric_cols,
        var_name='metric_fold',
        value_name='value'
    )
    
    # Extract metric name and fold number using regex pattern
    pattern = r'metric_(.+?)_fold_(\d+)$'
    extracted = melted_df['metric_fold'].str.extract(pattern)
    
    # Create new columns
    melted_df['metric'] = extracted[0]
    melted_df['fold'] = extracted[1]
    
    # For metrics without fold numbers (like 'metric_total_training_time')
    # Fill NaN metric names with the original string without 'metric_' prefix
    mask = melted_df['metric'].isna()
    melted_df.loc[mask, 'metric'] = melted_df.loc[mask, 'metric_fold'].str.replace('metric_', '')
    
    # Drop the temporary column and clean up
    melted_df = melted_df.drop('metric_fold', axis=1)
    melted_df = melted_df.reset_index(drop=True)
    
    return melted_df

In [34]:
def prepare_analogue(name):
    analogue_df = filter_data(analogues[name])
    ERROR_COLS = [f'fold_{i}_error' for i in range(5)]
    errors = analogue_df[ERROR_COLS].isna().all(axis=1)
    return melt_metrics(analogue_df[errors]).rename(columns={'learning_rate': 'lr'}).groupby(['dataset', 'subsample', 'lr', 'metric']).agg({'value': 'mean'})


In [35]:
processed

{'sigmoid_5.2.1':               dataset sketch_method sketch_outputs smoothing_alpha  \
 0      age_prediction          topk              1             0.9   
 1      age_prediction          topk              1             0.9   
 2      age_prediction          topk              1             0.9   
 3      age_prediction          topk              1             0.9   
 4      age_prediction          topk              1             0.9   
 ...               ...           ...            ...             ...   
 11045           yeast          topk              7             0.9   
 11046           yeast          topk              7             0.9   
 11047           yeast          topk              7             0.9   
 11048           yeast          topk              7             0.9   
 11049           yeast          topk              7             0.9   
 
       stabilization_threshold subsample     lr  \
 0                         0.5      0.05  0.005   
 1                         

In [36]:
def prepare_modified(name):
    results = processed[name].reset_index().groupby(['dataset', 'subsample', 'lr', 'metric']).agg({'value': ['max', 'min']})
    results.columns = results.columns.to_flat_index()
    return results 

In [37]:
def comparison_analogues(modified_name, analogue_name, analogue_model='catboost'):
    if not analogue_model:
        analogue_model = analogue_name
    modified_results = prepare_modified(modified_name)
    join_data = prepare_analogue(analogue_name).join(modified_results).reset_index()
    join_data.dropna(axis=0, inplace=True)
    cols = list(join_data.columns)
    cols[-2:]  = ['max', 'min']
    join_data.columns = cols
    join_data[modified_name] = join_data['max']
    join_data = join_data.rename(columns={'value': analogue_model})
    comp_metrics = join_data.metric.isin(COMPUTATIONAL_METRICS)
    join_data.loc[comp_metrics, modified_name] = join_data.loc[comp_metrics, 'min']
    join_data.drop(['max', 'min'], axis=1, inplace=True)
    change_col = f'{modified_name}_over_{analogue_name}'
    join_data[change_col] = (join_data[modified_name] - join_data[analogue_model]) * 100
    join_data[change_col][comp_metrics] = join_data[change_col][comp_metrics] / join_data[analogue_model]
    is_better = ((join_data[change_col] > 0) & (~comp_metrics) ).map({False: '-', True: '+'})
    join_data['is_better'] = is_better
    return join_data


comp = comparison_analogues('sigmoid_3.1', 'analogues_3.1')
comp_lgbm = comparison_analogues('sigmoid_3.1', 'analogues_lgbm_3.1')

KeyError: 'sigmoid_3.1'

In [255]:
comp = comp[~comp.metric.isin(['mean_nodes', 'mean_leaves'])]
comp_lgbm = comp_lgbm[~comp_lgbm.metric.isin(['mean_nodes', 'mean_leaves'])]

In [251]:
COMPARISON_DIR = Path('comparison_analogues')
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

In [252]:
comp.drop('dataset', axis=1).to_csv(COMPARISON_DIR / 'comparison_age_pred_to_catboost_sigmoid.csv')

In [ ]:
prepare_analogue('analogues_3.1').join(results)

value  (value, max)  \
dataset        subsample lr   metric                                     
age_prediction 0.05      0.05 accuracy          0.297500      0.374600   
                              f1                0.213951      0.363259   
                              inference_time   11.021197     18.103744   
                              mean_leaves       0.000000     30.061639   
                              mean_nodes        0.000000    127.000000   
...                                                  ...           ...   
yeast          0.75      0.1  mean_leaves       0.000000           NaN   
                              mean_nodes        0.000000           NaN   
                              precision         0.704321           NaN   
                              recall            0.483847           NaN   
                              train_time      249.887582           NaN   

                                              (value, min)  
dataset        subsample lr   metric                        
age_prediction 0.05      0.05 accuracy            0.356167  
                              f1                  0.351285  
                              inference_time     13.142054  
                              mean_leaves        25.666699  
                              mean_nodes        127.000000  
...                                                    ...  
yeast          0.75      0.1  mean_leaves              NaN  
                              mean_nodes               NaN  
                              precision                NaN  
                              recall                   NaN  
                              train_time               NaN  

[432 rows x 3 columns]

## LaTex formation

In [26]:
cat = pd.concat([df.reset_index() for df in
    processed.values()
], axis=0)

In [27]:
humming_loss = cat[(cat.metric == 'accuracy')].copy() 
humming_loss['value'] = 1 - humming_loss['value']
humming_loss['metric'] = 'hamming_loss'

In [28]:
cat = pd.concat(
    [cat, humming_loss], axis=0
)

In [20]:
METRICS_TO_EXTRACT = [
    'train_time', 
    'inference_time', 
    'f1',
    'f1_macro',
    'f1_micro',
    'humming_loss'
]

DATASETS_TO_EXTRACT = [
    'genbase',
    'yeast', 
    'mnist',
    'age_prediction',
    'mediamill',
    'cifar10'
]

EXPERIMENTAL_ORDER = [
    'sigmoid_5.2.1',
    'hyperbolic_5.2.1',
    'baselines_5.2.1',
    'lgbm_5.2.1',
    'xgboost_5.2.1'
]

In [21]:
TO_MAX = [
    'f1',
    'f1_macro',
    'f1_micro',
]
TO_MIN = [m for m in METRICS_TO_EXTRACT if m not in TO_MAX]

OURS = [
    'sigmoid_5.2.1',
    'hyperbolic_5.2.1'
]

In [22]:

HP_SETUP = {
    'stabilization_threshold': '1.0',
    'lr': "0.1"
}

TARGET_DIMENSIONS = [
    'dataset', 'experiment', 'metric'
]

filtered_total = cat[(cat.dataset.isin(DATASETS_TO_EXTRACT)) & (cat.metric.isin(METRICS_TO_EXTRACT))]

for c, val in HP_SETUP.items():
    filtered_total[c].fillna(val, inplace=True)
    filtered_total = filtered_total[filtered_total[c] == val]

filtered_total = filtered_total.drop([c for c in filtered_total if c not in TARGET_DIMENSIONS + ['value', 'std']], axis=1)

In [23]:
grp_total = filtered_total.groupby(TARGET_DIMENSIONS)

min_total = grp_total.aggregate('min').reset_index()
max_total = grp_total.aggregate('max').reset_index()
total = grp_total.aggregate('mean').reset_index()

In [24]:
idx = (total.experiment.isin(OURS)) & (total.metric.isin(TO_MAX))

total.loc[idx, 'value'] = max_total.loc[idx, 'value']
idx = (total.experiment.isin(OURS)) & (total.metric.isin(TO_MIN))
total.loc[idx, 'value'] = min_total.loc[idx, 'value']

total['std'] = min_total['std']

In [25]:
total

,dataset,experiment,metric,value,std
0,age_prediction,baselines_5.2.1,f1,0.363172,0.003967
1,age_prediction,baselines_5.2.1,f1_macro,0.362712,0.003960
2,age_prediction,baselines_5.2.1,f1_micro,0.373243,0.002727
3,age_prediction,baselines_5.2.1,inference_time,44.756055,2.880876
4,age_prediction,baselines_5.2.1,train_time,2329.729660,287.332018
...,...,...,...,...,...
130,yeast,xgboost_5.2.1,f1,0.514582,0.019844
131,yeast,xgboost_5.2.1,f1_macro,0.332574,0.011386
132,yeast,xgboost_5.2.1,f1_micro,0.535267,0.021500
133,yeast,xgboost_5.2.1,inference_time,13.582791,0.639641


In [28]:
total

,dataset,experiment,metric,value,std,formatted_value
0,age_prediction,baselines_5.2.1,f1,0.363172,0.003967,0.363±0.004
1,age_prediction,baselines_5.2.1,f1_macro,0.362712,0.003960,0.363±0.004
2,age_prediction,baselines_5.2.1,f1_micro,0.373243,0.002727,0.373±0.003
3,age_prediction,baselines_5.2.1,inference_time,44.756055,2.880876,44.756±2.881
4,age_prediction,baselines_5.2.1,train_time,2329.729660,287.332018,2329.730±287.332
...,...,...,...,...,...,...
130,yeast,xgboost_5.2.1,f1,0.514582,0.019844,0.515±0.020
131,yeast,xgboost_5.2.1,f1_macro,0.332574,0.011386,0.333±0.011
132,yeast,xgboost_5.2.1,f1_micro,0.535267,0.021500,0.535±0.022
133,yeast,xgboost_5.2.1,inference_time,13.582791,0.639641,13.583±0.640


In [26]:
import pandas as pd
import numpy as np

def format_paper_table(df, baseline_experiment, pm='±'):
    """
    Transform a dataframe into a formatted table for scientific papers.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe with columns: dataset, experiment, metric, value, std
    baseline_experiment : str
        The name of the experiment to use as baseline for change calculation
    
    Returns:
    --------
    pd.DataFrame
        Formatted dataframe with columns: dataset, metric, *experiments
        Each cell format: {value}\pm{std} / {change}
    """
    
    # Define metrics that use raw difference
    raw_diff_metrics = ['hamming_loss', 'f1', 'f1_macro', 'f1_micro']
    
    # Get all unique experiments
    experiments = df['experiment'].unique()
    
    # Separate baseline data
    baseline_df = df[df['experiment'] == baseline_experiment]
    
    # Pivot the dataframe to have experiments as columns
    # First, create a column with formatted value±std
    df['formatted_value'] = df.apply(
        lambda row: f"{row['value']:.3f}{pm}{row['std']:.3f}", 
        axis=1
    )
    
    # Pivot to get experiments as columns
    pivot_df = df.pivot_table(
        index=['dataset', 'metric'],
        columns='experiment',
        values='formatted_value',
        aggfunc='first'
    ).reset_index()
    
    # Calculate changes for each experiment relative to baseline
    for exp in experiments:
        if exp == baseline_experiment:
            continue
            
        # Merge baseline and current experiment data
        merged = df[df['experiment'] == exp].merge(
            baseline_df,
            on=['dataset', 'metric'],
            suffixes=('', '_baseline')
        )
        
        # Calculate change based on metric type
        changes = []
        for _, row in merged.iterrows():
            metric = row['metric']
            value = row['value']
            baseline = row['value_baseline']
            std = row['std']
            
            if metric in raw_diff_metrics:
                change = (value - baseline) * 100
                change_str = f"{change:+.3f}"
            else:
                if baseline != 0:
                    change = ((value - baseline) / baseline) * 100
                    change_str = f"{change:+.1f}%"
                else:
                    change_str = "N/A"
            
            changes.append(change_str)
        
        # Add change information to the formatted value
        # Create a mapping from (dataset, metric) to change
        change_map = dict(zip(
            zip(merged['dataset'], merged['metric']),
            changes
        ))
        
        # Update the formatted values to include change
        for idx, row in pivot_df.iterrows():
            dataset = row['dataset']
            metric = row['metric']
            key = (dataset, metric)
            
            if key in change_map:
                original_value = row[exp]
                pivot_df.at[idx, exp] = f"{original_value} / {change_map[key]}"
    
    # Ensure baseline column doesn't have change info
    if baseline_experiment in pivot_df.columns:
        pivot_df[baseline_experiment] = pivot_df[baseline_experiment].apply(
            lambda x: x.split(' / ')[0] if ' / ' in str(x) else x
        )
    
    # Sort columns: dataset, metric, then experiments
    experiment_cols = [col for col in pivot_df.columns if col not in ['dataset', 'metric']]
    # Sort experiments to have baseline first (optional)
    experiment_cols_sorted = [baseline_experiment] + sorted([e for e in experiment_cols if e != baseline_experiment])
    
    final_df = pivot_df[['dataset', 'metric'] + experiment_cols_sorted]
    # final_df.drop('experiment', axis=1, inplace=True)
    
    return final_df


In [31]:
os.getcwd()

'/home/leostre/Рабочий стол/py-boost/experiments'

In [32]:
format_paper_table(total, 'baselines_5.2.1')[[ 'dataset', 'metric', *EXPERIMENTAL_ORDER]]

experiment,dataset,metric,sigmoid_5.2.1,hyperbolic_5.2.1,baselines_5.2.1,lgbm_5.2.1,xgboost_5.2.1
0,age_prediction,f1,0.362±0.003 / -0.146,0.364±0.004 / +0.099,0.363±0.004,0.293±0.007 / -6.992,0.293±0.006 / -7.024
1,age_prediction,f1_macro,0.361±0.003 / -0.150,0.364±0.004 / +0.097,0.363±0.004,0.293±0.007 / -6.935,0.293±0.006 / -6.967
2,age_prediction,f1_micro,0.377±0.004 / +0.329,0.378±0.004 / +0.506,0.373±0.003,0.325±0.007 / -4.827,0.322±0.006 / -5.165
3,age_prediction,inference_time,19.101±0.873 / -57.3%,21.104±1.641 / -52.8%,44.756±2.881,126.862±1.324 / +183.5%,98.493±2.441 / +120.1%
4,age_prediction,train_time,7555.158±454.636 / +224.3%,5833.342±406.882 / +150.4%,2329.730±287.332,4031.379±35.788 / +73.0%,8286.741±138.380 / +255.7%
5,cifar10,f1,0.440±0.004,NaN,NaN,0.259±0.006,0.249±0.005
6,cifar10,f1_macro,0.441±0.004,NaN,NaN,0.260±0.006,0.250±0.005
7,cifar10,f1_micro,0.444±0.005,NaN,NaN,0.246±0.006,0.239±0.007
8,cifar10,inference_time,12.138±0.260,NaN,NaN,119.533±1.988,53.106±1.390
9,cifar10,train_time,1008.633±16.469,NaN,NaN,6602.212±86.107,20011.505±90.475


In [286]:
cat[(cat.dataset == 'genbase') & (cat.metric == 'f1_micro') & (cat.experiment == 'xgboost_5.2.1')]

,dataset,sketch_method,sketch_outputs,smoothing_alpha,stabilization_threshold,subsample,lr,metric,value,std,experiment
296,genbase,NaN,NaN,NaN,NaN,0.05,0.005,f1_micro,0.0,NaN,xgboost_5.2.1
312,genbase,NaN,NaN,NaN,NaN,0.05,0.1,f1_micro,0.0,NaN,xgboost_5.2.1
328,genbase,NaN,NaN,NaN,NaN,0.5,0.005,f1_micro,0.0,NaN,xgboost_5.2.1
344,genbase,NaN,NaN,NaN,NaN,0.5,0.1,f1_micro,0.0,NaN,xgboost_5.2.1
360,genbase,NaN,NaN,NaN,NaN,0.75,0.005,f1_micro,0.0,NaN,xgboost_5.2.1
376,genbase,NaN,NaN,NaN,NaN,0.75,0.1,f1_micro,0.0,NaN,xgboost_5.2.1


In [287]:
import pandas as pd
import numpy as np
from scipy import stats
import math
from typing import Dict, List, Tuple, Optional
from statsmodels.stats.multitest import multipletests

def statistical_comparison(
    df: pd.DataFrame,
    baseline_experiment: str,
    alpha: float = 0.05,
    correction_method: str = 'bonferroni',
    test_type: str = 'independent',
    n_samples: Optional[Dict[str, int]] = None
) -> pd.DataFrame:
    """
    Perform statistical comparisons between experiments and baseline for each dataset and metric.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe with columns: dataset, experiment, metric, value, std
    baseline_experiment : str
        Name of the baseline experiment to compare against
    alpha : float
        Significance level (default: 0.05)
    correction_method : str
        Multiple comparison correction method: 'bonferroni', 'fdr', 'holm', or None
    test_type : str
        Type of test: 'independent' (two-sample t-test) or 'paired' (paired t-test)
    n_samples : dict
        Dictionary mapping experiment names to number of samples/repetitions.
        If None, assumes all experiments have same number of samples.
    
    Returns:
    --------
    pd.DataFrame
        Summary dataframe with statistical test results for each comparison
    """
    
    results = []
    
    # Get unique datasets and metrics
    datasets = df['dataset'].unique()
    metrics = df['metric'].unique()
    experiments = df['experiment'].unique()
    
    # Separate baseline data
    baseline_df = df[df['experiment'] == baseline_experiment]
    
    # If n_samples not provided, estimate from data (assuming same for all)
    if n_samples is None:
        n_samples = {}
        for exp in experiments:
            # Estimate sample size from std (if you have raw data, you'd get actual n)
            # This is a placeholder - in practice, you should provide actual sample sizes
            exp_data = df[df['experiment'] == exp]
            # Assuming std is from N samples, you need to know N
            # For demonstration, assuming N=30 if not provided
            n_samples[exp] = 30
    
    # Collect all p-values for multiple comparison correction
    all_pvalues = []
    comparison_info = []
    
    # For each dataset and metric combination
    for dataset in datasets:
        for metric in metrics:
            # Get baseline data for this dataset/metric
            baseline_data = baseline_df[
                (baseline_df['dataset'] == dataset) & 
                (baseline_df['metric'] == metric)
            ]
            
            if len(baseline_data) == 0:
                continue
                
            baseline_value = baseline_data['value'].iloc[0]
            baseline_std = baseline_data['std'].iloc[0]
            baseline_n = n_samples.get(baseline_experiment, 30)
            
            # Compare with each other experiment
            for exp in experiments:
                if exp == baseline_experiment:
                    continue
                    
                # Get experiment data
                exp_data = df[
                    (df['dataset'] == dataset) & 
                    (df['metric'] == metric) & 
                    (df['experiment'] == exp)
                ]
                
                if len(exp_data) == 0:
                    continue
                    
                exp_value = exp_data['value'].iloc[0]
                exp_std = exp_data['std'].iloc[0]
                exp_n = n_samples.get(exp, 30)
                
                # Calculate difference and relative change
                raw_diff = exp_value - baseline_value
                if baseline_value != 0:
                    rel_change = (raw_diff / baseline_value) * 100
                else:
                    rel_change = np.nan
                
                # Perform statistical test
                if test_type == 'independent':
                    t_stat, p_value = independent_t_test(
                        baseline_value, baseline_std, baseline_n,
                        exp_value, exp_std, exp_n
                    )
                    test_name = "Independent t-test"
                elif test_type == 'paired':
                    # For paired test, you need the actual paired differences
                    # This is a simplified version assuming you have the difference std
                    # In practice, you'd need the raw paired data
                    t_stat, p_value = paired_t_test_approx(
                        raw_diff, baseline_std, exp_std, baseline_n
                    )
                    test_name = "Paired t-test (approximated)"
                else:
                    raise ValueError(f"Unknown test_type: {test_type}")
                
                # Store for multiple comparison correction
                all_pvalues.append(p_value)
                comparison_info.append({
                    'dataset': dataset,
                    'metric': metric,
                    'experiment': exp,
                    'baseline': baseline_experiment,
                    'baseline_value': baseline_value,
                    'baseline_std': baseline_std,
                    'exp_value': exp_value,
                    'exp_std': exp_std,
                    'raw_diff': raw_diff,
                    'rel_change': rel_change,
                    't_statistic': t_stat,
                    'p_value': p_value,
                    'test_type': test_name,
                    'baseline_n': baseline_n,
                    'exp_n': exp_n
                })
    
    # Apply multiple comparison correction
    if correction_method and all_pvalues:
        reject, pvals_corrected, _, _ = multipletests(
            all_pvalues, 
            alpha=alpha, 
            method=correction_method
        )
        
        # Add corrected p-values and significance flags
        for i, info in enumerate(comparison_info):
            info['p_value_corrected'] = pvals_corrected[i]
            info['significant_raw'] = info['p_value'] < alpha
            info['significant_corrected'] = pvals_corrected[i] < alpha
            
            # Add significance stars
            if pvals_corrected[i] < 0.05:
                info['significance'] = ''
            else:
                info['significance'] = '*'
    
    # Create results dataframe
    results_df = pd.DataFrame(comparison_info)
    
    return results_df


def independent_t_test(
    mean1: float, std1: float, n1: int,
    mean2: float, std2: float, n2: int
) -> Tuple[float, float]:
    """
    Perform independent two-sample t-test (Welch's t-test).
    
    Returns:
    --------
    tuple: (t_statistic, p_value)
    """
    # Standard error
    se = math.sqrt((std1**2 / n1) + (std2**2 / n2))
    
    # t-statistic
    t_stat = (mean2 - mean1) / se
    
    # Degrees of freedom (Welch-Satterthwaite)
    df = ((std1**2 / n1 + std2**2 / n2)**2) / \
         ((std1**2 / n1)**2 / (n1 - 1) + (std2**2 / n2)**2 / (n2 - 1))
    
    # p-value (two-tailed)
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df))
    
    return t_stat, p_value


def paired_t_test_approx(
    mean_diff: float,
    std1: float,
    std2: float,
    n: int,
    correlation: float = 0.5
) -> Tuple[float, float]:
    """
    Approximate paired t-test when only means and stds are available.
    This requires an estimate of the correlation between paired measurements.
    
    Parameters:
    -----------
    mean_diff : float
        Mean of differences (exp - baseline)
    std1, std2 : float
        Standard deviations of the two groups
    n : int
        Number of pairs
    correlation : float
        Estimated correlation between paired measurements (default: 0.5)
    
    Returns:
    --------
    tuple: (t_statistic, p_value)
    """
    # Standard deviation of differences
    std_diff = math.sqrt(std1**2 + std2**2 - 2 * correlation * std1 * std2)
    
    # Standard error of the differences
    se = std_diff / math.sqrt(n)
    
    # t-statistic
    t_stat = mean_diff / se
    
    # Degrees of freedom
    df = n - 1
    
    # p-value (two-tailed)
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df))
    
    return t_stat, p_value


def add_statistical_significance_to_table(
    formatted_df: pd.DataFrame,
    stats_results: pd.DataFrame,
    show_corrected: bool = True
) -> pd.DataFrame:
    """
    Add significance indicators to the formatted paper table.
    
    Parameters:
    -----------
    formatted_df : pd.DataFrame
        The formatted table from format_paper_table()
    stats_results : pd.DataFrame
        Results from statistical_comparison()
    show_corrected : bool
        Use corrected p-values for significance indicators
    
    Returns:
    --------
    pd.DataFrame with significance indicators added
    """
    
    result_df = formatted_df.copy()
    
    # Create a lookup dictionary for significance
    sig_lookup = {}
    for _, row in stats_results.iterrows():
        key = (row['dataset'], row['metric'], row['experiment'])
        if show_corrected:
            sig_lookup[key] = row['significance']
        else:
            if row['significant_raw']:
                if row['p_value'] < 0.05:
                    sig_lookup[key] = ''
                else:
                    sig_lookup[key] = '*'
            else:
                sig_lookup[key] = '*'
    
    # Add significance to each cell
    for idx, row in result_df.iterrows():
        dataset = row['dataset']
        metric = row['metric']
        
        for col in result_df.columns:
            if col not in ['dataset', 'metric']:
                exp = col
                key = (dataset, metric, exp)
                
                if key in sig_lookup and sig_lookup[key]:
                    cell_value = row[col]
                    result_df.at[idx, col] = f"{cell_value}{sig_lookup[key]}"
    
    return result_df
    
# Define sample sizes (number of runs/repetitions)

from collections import defaultdict
n_samples = defaultdict(lambda x: 5)

    
    # Perform statistical comparisons
stats_results = statistical_comparison(
        df=total,
        baseline_experiment='baselines_5.2.1',
        alpha=0.05,
        correction_method='bonferroni',
        test_type='independent',
        n_samples=n_samples
)
    


In [288]:

formatted_table = format_paper_table(total, baseline_experiment='baselines_5.2.1')
final_table = add_statistical_significance_to_table(formatted_table, stats_results)
    

In [289]:
final_table

experiment,dataset,metric,baselines_5.2.1,hyperbolic_5.2.1,lgbm_5.2.1,sigmoid_5.2.1,xgboost_5.2.1
0,age_prediction,f1,0.363±0.004,0.364±0.004 / +0.099*,0.293±0.007 / -6.992,0.362±0.003 / -0.146*,0.293±0.006 / -7.024
1,age_prediction,f1_macro,0.363±0.004,0.364±0.004 / +0.097*,0.293±0.007 / -6.935,0.361±0.003 / -0.150*,0.293±0.006 / -6.967
2,age_prediction,f1_micro,0.373±0.003,0.378±0.004 / +0.506,0.325±0.007 / -4.827,0.377±0.004 / +0.329*,0.322±0.006 / -5.165
3,age_prediction,humming_loss,0.627±0.003,0.622±0.004 / -0.506,0.675±0.007 / +4.827,0.623±0.004 / -0.329*,0.678±0.006 / +5.165
4,age_prediction,inference_time,44.756±2.881,21.104±1.641 / -52.8%,126.862±1.324 / +183.5%,19.101±0.873 / -57.3%,98.493±2.441 / +120.1%
5,age_prediction,train_time,2329.730±287.332,5833.342±406.882 / +150.4%,4031.379±35.788 / +73.0%,7555.158±454.636 / +224.3%,8286.741±138.380 / +255.7%
6,cifar10,f1,NaN,NaN,0.259±0.006,NaN,0.249±0.005
7,cifar10,f1_macro,NaN,NaN,0.260±0.006,NaN,0.250±0.005
8,cifar10,f1_micro,NaN,NaN,0.246±0.006,NaN,0.239±0.007
9,cifar10,humming_loss,NaN,NaN,0.754±0.006,NaN,0.761±0.007


## Ablations 



In [ ]:
cat.drop('std', axis=1)

,dataset,sketch_method,sketch_outputs,smoothing_alpha,stabilization_threshold,subsample,lr,metric,value,std,experiment
0,age_prediction,topk,1,0.9,0.5,0.05,0.005,accuracy,0.386233,0.005516,sigmoid_5.2.1
1,age_prediction,topk,1,0.9,0.5,0.05,0.005,adaptive_threshold_applied,NaN,NaN,sigmoid_5.2.1
2,age_prediction,topk,1,0.9,0.5,0.05,0.005,adaptive_threshold_fallback,NaN,NaN,sigmoid_5.2.1
3,age_prediction,topk,1,0.9,0.5,0.05,0.005,adaptive_threshold_inference_time,NaN,NaN,sigmoid_5.2.1
4,age_prediction,topk,1,0.9,0.5,0.05,0.005,bce_loss,NaN,NaN,sigmoid_5.2.1
...,...,...,...,...,...,...,...,...,...,...,...
688,yeast,NaN,NaN,NaN,NaN,0.05,0.1,humming_loss,0.495954,0.021500,xgboost_5.2.1
704,yeast,NaN,NaN,NaN,NaN,0.5,0.005,humming_loss,0.449450,0.023799,xgboost_5.2.1
720,yeast,NaN,NaN,NaN,NaN,0.5,0.1,humming_loss,0.455526,0.029293,xgboost_5.2.1
736,yeast,NaN,NaN,NaN,NaN,0.75,0.005,humming_loss,0.437990,0.030608,xgboost_5.2.1


In [62]:
create_interactive_metric_scatter(cat.drop('std', axis=1), 'inference_time', 'f1', signature_flag=True)

NameError: name 'create_interactive_metric_scatter' is not defined

## Исследование времени операций

In [172]:
hp = processed['sigmoid_exp_smoothing_4.0'].reset_index().dropna()

In [173]:
is_time_metric = hp.metric.map(lambda x: x.endswith('_time'))

In [174]:
hp[(is_time_metric) & (hp.metric != 'inference_time')]

,dataset,sketch_method,sketch_outputs,subsample,lr,metric,value
514,mnist,topk,1,0.05,0.005,get_indexers_avg_time,0.734327
516,mnist,topk,1,0.05,0.005,get_indexers_total_time,11.149722
517,mnist,topk,1,0.05,0.005,get_weights_avg_time,0.276638
519,mnist,topk,1,0.05,0.005,get_weights_total_time,4.202707
527,mnist,topk,1,0.05,0.005,train_time,1214.177747
...,...,...,...,...,...,...,...
1010,mnist,topk,7,0.75,0.1,get_indexers_avg_time,0.787379
1012,mnist,topk,7,0.75,0.1,get_indexers_total_time,180.807680
1013,mnist,topk,7,0.75,0.1,get_weights_avg_time,0.270688
1015,mnist,topk,7,0.75,0.1,get_weights_total_time,62.406023


In [175]:
tt = hp[hp.metric == 'train_time'].groupby(['sketch_outputs', 'subsample', 'lr']).value.aggregate('first')
git = hp[hp.metric == 'get_indexers_total_time'].groupby(['sketch_outputs', 'subsample', 'lr']).value.aggregate('first')
gwt = hp[hp.metric == 'get_weights_total_time'].groupby(['sketch_outputs', 'subsample', 'lr']).value.aggregate('first')

In [176]:
git.name = 'get_indexers'
gwt.name = 'get_weights'

In [177]:
props = pd.concat([
    git / tt * 100,
    gwt / tt * 100
], axis=1)
props.columns = ['get_indexers', 'get_weights']


In [180]:
props.reset_index().groupby(['sketch_outputs', 'subsample',]).agg({'get_indexers': ['max', 'min'], 'get_weights': ['max', 'min']}).round(2)

get_indexers       get_weights      
                                  max   min         max   min
sketch_outputs subsample                                     
1              0.05              1.74  0.92        0.67  0.35
               0.25              2.51  1.58        1.01  0.62
               0.5               2.87  2.08        0.99  0.72
               0.75              2.83  2.38        0.98  0.83
2              0.05              2.03  0.93        0.80  0.39
               0.25              2.49  1.85        1.00  0.73
               0.5               2.87  2.13        0.98  0.71
               0.75              2.85  2.31        0.99  0.79
5              0.05              2.26  0.93        0.92  0.34
               0.25              2.52  2.27        1.01  0.91
               0.5               2.90  2.61        1.00  0.89
               0.75              2.89  2.64        0.99  0.91
7              0.05              2.27  0.94        0.89  0.37
               0.25              2.53  2.26        1.03  0.90
               0.5               2.88  2.57        1.01  0.87
               0.75              2.91  2.53        1.01  0.87

In [ ]:
hp[(is_time_metric) & (hp.metric != 'inference_time')]

In [85]:
import seaborn as sns 
import matplotlib.pyplot as plt 


lgbm_df = df.groupby(['pred_thr', 'prop_keep', 'metric'])['value'].aggregate('mean').reset_index()

# Pivot the data to create a matrix for each metric
pivot_dfs = {}
metrics = df['metric'].unique()

for metric in metrics:
    # Filter for the specific metric
    metric_df = lgbm_df[lgbm_df['metric'] == metric].copy()
    
    # Pivot to create matrix
    pivot = metric_df.pivot_table(
        index='pred_thr', 
        columns='prop_keep', 
        values='value',
        aggfunc='first'
    )
    pivot_dfs[metric] = pivot

# Create heatmaps for each metric
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (metric, pivot_df) in enumerate(pivot_dfs.items()):
    if i < len(axes):
        sns.heatmap(pivot_df, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[i],
                   cbar_kws={'label': metric})
        axes[i].set_title(f'{metric} Heatmap')
        axes[i].set_xlabel('prop_keep')
        axes[i].set_ylabel('pred_thr')

# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

# Create separate, more detailed heatmaps for each metric
for metric, pivot_df in pivot_dfs.items():
    plt.figure(figsize=(8, 6))
    
    # Use different colormaps based on metric type
    if metric in ['inference_time', 'train_time']:
        cmap = 'viridis_r'  # Reverse viridis for time (darker = better)
    elif metric in ['accuracy', 'f1', 'precision', 'recall', 'roc_auc']:
        cmap = 'YlOrRd'  # Yellow-Orange-Red for performance metrics
    else:
        cmap = 'coolwarm'
    
    sns.heatmap(pivot_df, annot=True, fmt='.3f', cmap=cmap,
               cbar_kws={'label': metric}, linewidths=1, linecolor='gray')
    plt.title(f'{metric} - pred_thr vs prop_keep', fontsize=14, fontweight='bold')
    plt.xlabel('prop_keep', fontsize=12)
    plt.ylabel('pred_thr', fontsize=12)
    plt.tight_layout()
    plt.show()

# Print summary statistics
print("\n" + "="*50)
print("SUMMARY STATISTICS BY METRIC")
print("="*50)

for metric, pivot_df in pivot_dfs.items():
    print(f"\n{metric.upper()}:")
    print(f"  Best value: {pivot_df.max().max():.6f}")
    print(f"  Worst value: {pivot_df.min().min():.6f}")
    
    if metric in ['accuracy', 'f1', 'precision', 'recall', 'roc_auc']:
        # For performance metrics, find best combination
        best_loc = np.unravel_index(pivot_df.values.argmax(), pivot_df.shape)
        print(f"  Best combination: pred_thr={pivot_df.index[best_loc[0]]}, prop_keep={pivot_df.columns[best_loc[1]]}")
    elif metric in ['inference_time', 'train_time']:
        # For time metrics, lower is better
        best_loc = np.unravel_index(pivot_df.values.argmin(), pivot_df.shape)
        print(f"  Best combination (fastest): pred_thr={pivot_df.index[best_loc[0]]}, prop_keep={pivot_df.columns[best_loc[1]]}")

KeyError: 'pred_thr'

In [34]:
processed.keys()

dict_keys(['sigmoid_5.2.1', 'hyperbolic_5.2.1', 'baselines_5.2.1', 'lgbm_5.2.1', 'xgboost_5.2.1'])